# Identifikasi bahasa

Tanpa gold, laporkan estimasi, ketidakpastian dan audit campuran bahasa. Jika ada label bahasa gold, gunakan pipeline supervised tambahan.

Kode lengkap dari pipeline dan snippet asli. Ikuti urutan tahap dan ubah konfigurasi awal sesuai data. Alternatif tersedia pada cheatsheet utama.

## Setup & konfigurasi

Satu-satunya sel yang wajib diubah. Jalankan paling awal.

In [ ]:
import os
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", TOKENIZERS_PARALLELISM="false")
from pathlib import Path
import importlib.metadata as md

def env_report(models_dir=None):
    import torch
    for p in ("torch", "transformers", "scikit-learn", "pandas", "numpy", "sentence-transformers", "lightgbm"):
        try:
            print(p, md.version(p))
        except md.PackageNotFoundError:
            print(p, "TIDAK ADA")
    gpu = torch.cuda.is_available()
    print("cuda", gpu, torch.cuda.get_device_name(0) if gpu else "")
    if gpu:
        print("VRAM GB", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "| bf16", torch.cuda.is_bf16_supported())
    if models_dir and Path(models_dir).exists():
        print("model lokal:", sorted(p.name for p in Path(models_dir).iterdir() if p.is_dir()))
    return gpu

import os
import random
import numpy as np
import torch

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

import time
from datetime import datetime
import numpy as np

GATES = [(40, "G1 pilih cabang task"), (70, "G2 submission pertama"), (105, "G3 lanjut atau lewati BERT"), (150, "G4 proyeksi BERT"), (225, "G5 blend"), (240, "Freeze model"), (280, "Unggah final"), (300, "Lomba selesai")]
_T0 = [time.time()]

def start_clock(hhmm=None):
    """hhmm='13:00' saat lomba. Tanpa argumen, hitung dari sekarang."""
    if hhmm:
        h, m = map(int, hhmm.split(":"))
        _T0[0] = datetime.now().replace(hour=h, minute=m, second=0, microsecond=0).timestamp()
    else:
        _T0[0] = time.time()

def clock(label=""):
    t = (time.time() - _T0[0]) / 60
    nxt = next((g for g in GATES if g[0] > t), None)
    print(f"[T+{t:.0f} menit] {label}" + (f" | berikutnya T{nxt[0]}: {nxt[1]}, sisa {nxt[0] - t:.0f} menit" if nxt else " | waktu habis"))
    return t

def scale_plan(n_rows, experiments, n_folds, medium=100_000, large=250_000, force=False):
    """Kembalikan (experiments, n_folds) yang aman untuk ukuran data. force=True mematikan pagar."""
    if force or n_rows <= medium:
        return experiments, n_folds
    if n_rows <= large:
        lighter = list(dict.fromkeys(("tfidf_word" if v in ("tfidf_union", "tfidf_char") else v, "sgd" if m in ("svc", "svc_cal", "rf", "mlp", "xgb", "lgbm") else m) for v, m in experiments))
        print(f"PAGAR SKALA {n_rows} baris: fitur char dibuang, {n_folds} fold jadi 3 -> {lighter}")
        return lighter, min(n_folds, 3)
    plan = [("hashing", "sgd"), ("tfidf_word", "sgd")]
    print(f"PAGAR SKALA {n_rows} baris: pakai {plan}, 3 fold. force=True bila yakin RAM dan waktu cukup.")
    return plan, min(n_folds, 3)

def sample_idx(n, k=100_000, seed=42):
    """Indeks acak untuk langkah mahal seperti bootstrap. Bila n <= k, semua baris."""
    return np.arange(n) if n <= k else np.random.default_rng(seed).choice(n, k, replace=False)

# ===== jalankan =====
import re, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
MODELS = Path(os.environ.get("COMP_ROOT", r"C:\gemastik")) / "models"
SEED = 42
seed_everything(SEED)
env_report(MODELS)
RACE_START = None                                # "13:00" saat lomba: jam menghitung gate dari sini

TRAIN, TEST, SAMPLE = "train.csv", "test.csv", "sample_submission.csv"
TEXT, TARGET, TIME = "text", "label", None      # TIME: nama kolom waktu bila ada
PREPROCESS = "tfidf"                             # none, tfidf, social, topic, transformer, indobertweet
EXPERIMENTS = [("tfidf_union", "lr"), ("tfidf_union", "svc"), ("tfidf_word", "cnb")]   # (vectorizer, model): tukar sesuka hati
FOLD_KIND, N_FOLDS = "auto", 5                   # auto, stratified, strat_group, group, time, source
USE_BERT = False                                 # True bila pretrained boleh dan lolos gerbang T105
BERT_MODEL, BERT_ROWS, MAX_LEN, BATCH = MODELS / "indobertweet", 50000, 128, 32
RESUME = False                                   # True: lanjut dari checkpoint terakhir setelah crash
CHECK_LABEL = False                              # True bila pretrained boleh: lacak sumber label
MBG_INSIGHT = True                               # bahan slide: aspek x sentimen, lokasi, tren
start_clock(RACE_START)


## Muat data

Baca file, otomatis mencoba encoding, simpan parquet.

In [ ]:
from pathlib import Path
import pandas as pd

def read_any(path, cache=True, **kw):
    """df = read_any('train.csv'). Lain kali otomatis membaca train.parquet."""
    p = Path(path)
    if p.suffix == ".parquet":
        return pd.read_parquet(p, **kw)
    if cache and p.with_suffix(".parquet").exists():
        return pd.read_parquet(p.with_suffix(".parquet"))
    if p.suffix in (".jsonl", ".json"):
        return pd.read_json(p, lines=p.suffix == ".jsonl", **kw)
    if p.suffix in (".xlsx", ".xls"):
        return pd.read_excel(p, **kw)
    sep = kw.pop("sep", "\t" if p.suffix == ".tsv" else ",")
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            df = pd.read_csv(p, sep=sep, encoding=enc, on_bad_lines="warn", **kw)
            break
        except UnicodeDecodeError:
            continue
    if cache:
        df.to_parquet(p.with_suffix(".parquet"), index=False)
    return df

# ===== jalankan =====
train, test = read_any(TRAIN), read_any(TEST)
print("train", train.shape, "| test", test.shape)
print(train.head(3).to_string())
EXPERIMENTS, N_FOLDS = scale_plan(len(train), EXPERIMENTS, N_FOLDS)   # data besar: otomatis lebih ringan


## EDA

Profil, kosong, duplikat, panjang, target, kata khas, OOV, overlap, adversarial, saran, dan porsi bahasa daerah.

In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_predict

def _nk(s):
    return s.fillna("").astype(str).str.lower().str.replace(r"\W+", " ", regex=True).str.strip()

def eda_report(train, test=None, text="text", target=None, time=None, n_show=8, sample=20000, seed=42):
    """Kembalikan dict ringkasan berisi saran dan rekomendasi max_words. target: nama kolom atau daftar kolom."""
    tips, out = [], {}
    print("=== PROFIL ===\ntrain", train.shape, "| test", None if test is None else test.shape)
    print(pd.DataFrame({"dtype": train.dtypes.astype(str), "null": train.isna().mean().round(4), "nunique": train.nunique()}).to_string())
    if test is not None:
        print("kolom hanya di train:", sorted(set(train.columns) - set(test.columns)))
    tx = train[text].fillna("").astype(str)
    print("\n=== CONTOH ===")
    for t in train[text].sample(min(n_show, len(train)), random_state=seed):
        print(" -", str(t)[:180])
    key = _nk(train[text])
    out["kosong"], out["duplikat"] = round(float(key.eq("").mean()), 4), round(float(key.duplicated().mean()), 4)
    print("\n=== KUALITAS ===\nkosong", out["kosong"], "| duplikat ternormalisasi", out["duplikat"], "| punya URL", round(tx.str.contains(r"https?://").mean(), 3),
          "| mention", round(tx.str.contains(r"@\w").mean(), 3), "| tanda baca akhir kalimat", round(tx.str.contains(r"[.!?]").mean(), 3))
    print("teks paling sering:", key.value_counts().head(3).to_dict())
    if out["duplikat"] > .01:
        tips.append("Duplikat > 1%: pakai group fold dan laporkan skor random serta grup.")
    w = tx.str.split().str.len()
    q = w.quantile([.5, .9, .95, .99]).round(0)
    out["max_words"] = int(q[.95])
    print("\n=== PANJANG (kata) ===\n", q.to_dict(), "| char p50", int(tx.str.len().median()))
    if q[.5] < 5:
        tips.append("Teks sangat pendek: utamakan TF-IDF karakter.")
    if q[.95] > 256:
        tips.append("Teks panjang: pertimbangkan head+tail dan chunk.")
    if target is not None:
        print("\n=== TARGET ===")
        cols = target if isinstance(target, (list, tuple)) else [target]
        if len(cols) > 1:
            Y = train[cols].to_numpy()
            print(pd.DataFrame({"pct": train[cols].mean().round(4)}).T.to_string(), "\nlabel per baris", round(float(Y.sum(1).mean()), 3))
        else:
            s = train[cols[0]]
            if pd.api.types.is_numeric_dtype(s) and s.nunique() > 20:
                print(s.describe().round(3).to_dict(), "skew", round(s.skew(), 3))
                if s.skew() > 1 and s.min() >= 0:
                    tips.append("Target berekor panjang: latih pada log1p.")
            else:
                vc = s.value_counts(dropna=False)
                print((vc / len(s)).round(4).to_dict())
                out["rasio_imbalance"] = round(float(vc.max() / max(vc.min(), 1)), 1)
                if out["rasio_imbalance"] > 20:
                    tips.append("Imbalance parah: metrik macro-F1/PR-AUC, decision layer dari OOF.")
                if s.nunique() <= 50 and s.notna().all():
                    v = TfidfVectorizer(max_features=30000, min_df=3, sublinear_tf=True)
                    X = v.fit_transform(tx.iloc[:sample])
                    m = LogisticRegression(C=4, max_iter=1000).fit(X, s.iloc[:sample].astype(str))
                    names, coef = v.get_feature_names_out(), m.coef_ if m.coef_.shape[0] > 1 else np.vstack([-m.coef_[0], m.coef_[0]])
                    print("kata khas (cek artefak):")
                    for j, c in enumerate(m.classes_):
                        print(f"  {c}:", [names[i] for i in coef[j].argsort()[::-1][:10]])
    cv = CountVectorizer(token_pattern=r"(?u)\b\w+\b", binary=True)
    X = cv.fit_transform(tx.sample(min(sample, len(tx)), random_state=seed))
    df_ = np.asarray(X.sum(0)).ravel()
    print("\n=== KOSAKATA ===\nvocab", X.shape[1], "| kata muncul sekali", round(float((df_ == 1).mean()), 3))
    if test is not None and text in test:
        te = test[text].fillna("").astype(str)
        vt = CountVectorizer(token_pattern=r"(?u)\b\w+\b").fit(te.sample(min(sample, len(te)), random_state=seed))
        cnt = np.asarray(vt.transform(te.sample(min(sample, len(te)), random_state=seed)).sum(0)).ravel()
        oov = np.array([x not in cv.vocabulary_ for x in vt.get_feature_names_out()])
        out["oov"] = round(float(cnt[oov].sum() / max(cnt.sum(), 1)), 3)
        tk = set(key)
        out["overlap"] = round(float(_nk(te).isin(tk).mean()), 4)
        print("OOV token test", out["oov"], "| teks test ada di train", out["overlap"])
        if out["oov"] > .2:
            tips.append("OOV tinggi: TF-IDF karakter atau FastText.")
        n = min(5000, len(tx), len(te))
        t2 = list(tx.sample(n, random_state=seed)) + list(te.sample(n, random_state=seed))
        lab = np.r_[np.zeros(n), np.ones(n)]
        Xa = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100000).fit_transform(t2)
        out["adversarial_auc"] = round(float(roc_auc_score(lab, cross_val_predict(LogisticRegression(max_iter=500), Xa, lab, cv=3, method="predict_proba")[:, 1])), 4)
        print("AUC adversarial", out["adversarial_auc"], "| panjang p50 test", int(te.str.split().str.len().median()))
        if out["adversarial_auc"] > .7:
            tips.append("Train dan test berbeda: cek kata pembeda, pertimbangkan split waktu/sumber.")
    if time and time in train:
        t = pd.to_datetime(train[time], errors="coerce")
        print("\n=== WAKTU ===\nrentang", t.min(), "s.d.", t.max(), "| gagal parse", round(float(t.isna().mean()), 3))
        if test is not None and time in test:
            tt = pd.to_datetime(test[time], errors="coerce")
            print("test", tt.min(), "s.d.", tt.max())
            if tt.min() > t.median():
                tips.append("Test lebih baru dari sebagian besar train: pakai time split.")
    out["saran"] = tips
    print("\n=== SARAN ===", *(tips or ["Tidak ada peringatan khusus."]), sep="\n - ")
    return out

import re
from collections import Counter
import numpy as np
import pandas as pd

# Kata yang sering di NusaX tiap bahasa dan jarang di korpus Indonesia acuan. Teruji pada NusaX test.
LANG_MARKERS = {
    "jav": "lan ing ning kanggo mangan uga nanging akeh panganane panganan dadi saka panggonan mung amarga kene rasane regane wis dhewe rega pelayanane marang uwis panggonane lak sek kuwi utawa panggon maneh luwih kanthi dinggo nganggo yaiku wektu uwong nggawe merga mbengi saiki panggone liyane kutha teka dina ngenteni kaluwarga arep adoh nganti mlebu dingge grapyak dalan awakdewe kanca entuk tuku siji nggarai mesthi mesen iwak suwi sega pithik mrene wengi kowe elek awan resik pemandangane omah njajal meh ben pilian ora ndak ndk mboten nggih ngono ngene awakmu piye durung wes karo ndang pancen ojo sampun panjenengan matur nuwun kulo sopo opo koyo enek duwit sego dewe podo seng lek kon ae wae",
    "sun": "sareng teu abdi ieu kanggo oge tiasa tuangeun tuang mung jeung seueur dieu sae dina tos kusabab tempatna upami rasana pikeun eta kuring tuangeunna hoyong deui cekap henteu arurang nyaeta cocog awis sumping wengi pangaos merenah mesen ayeuna sanes janten urang sawaktos rupi-rupi pangaosna sababaraha dinten anjeun teuing sapertos palayanan rerencangan pamandangan kulawargi kitu atawa ngeunah nganggo hayam bade keneh nepi leuwih keur mirah hungkul enggal mimiti kirang disarankeun dugi disayogikeun waktos tina ningali nguciwakeun lebet kedah jalmi tabuh sahiji atuh euy naon kumaha geus moal meni hatur nuhun kieu tong loba siga kunaon",
    "min": "untuak lamak indak bana ado tampek wak siko sangaik tampeknyo raso nio makanannyo harago cukuik samo rasonyo urang haragonyo karano juo labiah iko rancak pasan hanyo suko pamandangan biaso palayanan mambuek kaluarga dapek tibo piliahan paliang lua jauah buek ancak sadang masuak taruih sasuai sahinggo gadang baragam wakatu punyo manunggu lamo tajangkau sinan sajuak karajo disajian baliak salain patang mamasan kumuah talampau sanak sambia manuruik liak cubo basamo bantuak babarapo rami pai manikmati lueh apolai naiak mancari goreang direkomendasian dakek",
    "ban": "lan sane sajan ajengan ngajeng tiyang jaen nanging krana ten liu jaan tongos wenten tusing lakar utawi puniki sampun dados yening iraga anggon genah ajine ene uli luung dadi punika gen rasane nenten mael becik asane ajenganne tongosne sareng kanti buin ajengane niki suba dugas mesen ento antuk ngaenang lianan indik akeh driki dot peteng timpal ngranayang genahne jani ragane pengarga dina sekadi asapunika ngae anggen dogen teka nganggen kuciwa tiange taen meli makejang kuang bes patuh ngalih kasajiang enggal",
    "bjn": "kada gasan ulun banar jua kawa haja wadahnya handak ngini gin tagal haraga pang urang maulah haraganya risturan amun liwar katuju mayan palayanan ngitu cucuk ristu buhan pamandangan mun wayah pina kaluarga matan atawa mambari hawanya wahini sualnya kalu sasuai pian palayanannya labih hanyar babaya tajangkau rami parak hudah bamacam sakalinya imbah talalu mamasan lamahan babarapa tarus suah kadida kadada hancap sangkal pinanya muar malihat mahadang sigar sadang sabarataan rigat intang ikam disuruhakan bujur bubuhan manikmati katujuan jar gonol disurungakan",
    "mad": "bik ben engkok ghebey sarah kakanan ngakan riya tape rua mun bede kennengngan polana kiya kakananna rassana deri oreng deddhi terro pole cokop rassa bhei bhegus ongghu bekto diye regghena ghabay bennyak engko' deteng bennya' regghe olle pera' otabe mude lebur bekna senneng lebbi kennengnganna paleng ghuring selaen satiya pelean tadek kodhu kanca jhet settong pade kotta ta' messen korang kia entar bhuru aghebey pelayananna jhek ghik delem masok mare lerres acem-macem kenengngan ajem abit sampe' pendhenan padena lekkas bada",
    "ace": "hana ngon mangat lon nyoe pih keu nyan cit meunyo jeuet jih jak lagee watee pajoh mantong teumpat pah lom cukop kamoe ureueng get meuhai rumoh lam uroe saboh leubeh jeut teumpatjih lagak disinoe makananjih boh peunajoh atawa lheueh kon keuneuk rasajih loen leupah saweueb peue paleng ngen trok lee keunoe keuh seubab peulayanan droe sep sagai peuget keuluarga jinoe manok duek sabe phon ngoen inoe 'oh yumjih trep rakan preh peugah baro teungoh bagah tapajoh nakeuh lheuh gureng aneuk",
    "bug": "okko sibawa lao ladde' malunra' manre engka anre-anre ellinna aga pole rasana nasaba yako makanja' maneng maega okkoe anroang de' anroanna nasaba' iya' makanja iyaro restorang topa nappa wettu rupanna narekko makessing wenni onroang ladde manyameng anre mancaji makae kanja' iyye melo onroanna iyae pakkanreanna masempo kaminang makkokkoe masoli lunra' massuroang de'na wette malunra makkada maelo jaji tette' pajjamana okkoro laleng yaregga pirasai pakkanreang manu' mancajiwi iyanaritu esso deggaga sedding laingnge cedde' wettunna tennia taue sikenangeng runtu' meloki lettu' lettu",
    "bbc": "dohot dang tabo sipanganon jala hian adong inganan alai sian mansai gabe laho godang songon boi mangan naeng argana muse roha holan arga tung mambahen alana daina sahat manang dison parmanganan ikkon ingananna denggan hami akka lomo ima ahu tingki dope ndang borngin jolma dai bagak alani sipanganonna manuhor unang nai mura dalan sipilliton nungga hatop halak dongan uli sude boe panghobasionna lao panatapan marragam somal rap parjolo lambok tuson torus hurang sada iba dao sonang piga-piga menak lam panghobasion",
    "nij": "mangat eka tutu penginan diak kuman hong tege angat bahalap tuntang kea awi baya handak rega mawi hetuh hung ije uluh jituh beken amun panginan huang kilau ikei bujur andau dumah due puna hayak hindai wayah gin tame sukup hamalem lalau ewen labih meteh jatunti uras tempayah lime puluh mun haru sadang kawal hapan dinun manuk mengkeme ela kanih jatun intihan capat ampi telu munduk tukep nyerungan itah ikau belai sanang isut sinde pekasak mihup kurik dahang yuh ulih mite"
}
LANG_NAMES = {"ind": "Indonesia/lainnya", "jav": "Jawa", "sun": "Sunda", "min": "Minang", "ban": "Bali", "bjn": "Banjar", "mad": "Madura", "ace": "Aceh", "bug": "Bugis", "bbc": "Batak Toba", "nij": "Ngaju"}
_MARK = {k: set(v.split()) for k, v in LANG_MARKERS.items()}

def detect_local_language(text, min_hits=2):
    """min_hits=2 menjaga presisi. min_hits=1 untuk batas atas: lebih banyak salah."""
    w = set(re.findall(r"[a-zà-ÿ'’-]{3,}", str(text).lower()))
    best, hits = "ind", 0
    for code, m in _MARK.items():
        h = len(w & m)
        if h > hits:
            best, hits = code, h
    return best if hits >= min_hits else "ind"

def language_report(texts, show=5, seed=42):
    """Porsi tiap bahasa pada teks unik, batas ketat dan batas longgar, plus contoh untuk dibaca."""
    u = pd.Series(list(texts)).dropna().astype(str).drop_duplicates()
    strict, loose = u.map(detect_local_language), u.map(lambda t: detect_local_language(t, 1))
    tbl = pd.DataFrame({"ketat": strict.value_counts(), "longgar": loose.value_counts()}).fillna(0).astype(int)
    tbl.index = [LANG_NAMES.get(i, i) for i in tbl.index]
    tbl["ketat_%"] = (100 * tbl.ketat / len(u)).round(2)
    tbl["longgar_%"] = (100 * tbl.longgar / len(u)).round(2)
    local = float((strict != "ind").mean())
    upper = float((loose != "ind").mean())
    print(tbl.sort_values("longgar", ascending=False).to_string())
    print(f"Bahasa daerah: {local:.2%} (ketat) sampai {upper:.2%} (longgar) dari {len(u)} teks unik.")
    print("Saran:", "coba NusaBERT dan evaluasi per bahasa." if local >= .05 else "pantau dengan batas longgar; bila di atas 5%, baca sampel lalu coba NusaBERT." if upper >= .05 else "tidak perlu alur khusus; char n-gram cukup.")
    hit = u[strict != "ind"]
    for t in hit.sample(min(show, len(hit)), random_state=seed):
        print("  -", t[:100].replace("\n", " "))
    return strict.reindex(u.index)

# ===== jalankan =====
report = eda_report(train, test, TEXT, TARGET, time=TIME)
_ = language_report(train[TEXT].sample(min(100000, len(train)), random_state=SEED))
clock("EDA")


## Lacak sumber label

Kecocokan di atas 98% dengan satu model berarti label kemungkinan besar buatan model itu. Cek aturan dulu.

In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

def _agreement(pred, labels):
    """Cocokkan nama kelas lewat mayoritas, lalu hitung persetujuan. Aman walau label berupa angka atau bahasa lain."""
    p, y = pd.Series(np.asarray(pred).astype(str)), pd.Series(np.asarray(labels).astype(str))
    mapping = pd.crosstab(p, y).idxmax(axis=1)
    m = p.map(mapping)
    return float((m == y).mean()), float(cohen_kappa_score(m, y)), mapping.to_dict()

def _light_clean(t):
    t = re.sub(r"https?://\S+|@\w+|#", " ", str(t).lower())
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", t)).strip()

def _predict_model(path, texts, batch=64, max_len=128):
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(str(path), local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(str(path), local_files_only=True).eval()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(dev)
    order, out = np.argsort([len(t) for t in texts]), np.zeros(len(texts), dtype=int)
    with torch.inference_mode():
        for i in range(0, len(texts), batch):
            ix = order[i:i + batch]
            e = tok([texts[j] for j in ix], truncation=True, max_length=max_len, padding=True, return_tensors="pt").to(dev)
            out[ix] = model(**e).logits.argmax(1).cpu().numpy()
    del model
    return out

def _inset_score(texts, folder):
    """Skor InSet: jumlah bobot per kata, lalu tanda."""
    w = {}
    for f in ("positive.tsv", "negative.tsv"):
        d = pd.read_csv(Path(folder) / f, sep="\t")
        w.update(dict(zip(d.iloc[:, 0].astype(str), d.iloc[:, 1])))
    s = [sum(w.get(x, 0) for x in t.split()) for t in texts]
    return np.sign(s).astype(int)

def label_fingerprint(texts, labels, models_dir=None, names=("sent-mdhugol", "sent-w11wo", "sent-crypter70", "sent-aardiiiiy"), inset_dir=None, n=1500, seed=42):
    """Bandingkan label dengan calon sumber, pada teks mentah dan teks bersih. Kembalikan tabel terurut."""
    d = pd.DataFrame({"t": pd.Series(list(texts)).astype(str).to_numpy(), "y": list(labels)}).dropna()
    d = d.drop_duplicates("t").sample(min(n, len(d)), random_state=seed)
    views = {"mentah": d.t.tolist(), "bersih": [_light_clean(t) for t in d.t]}
    rows = []
    for name in names:
        p = Path(models_dir or "") / name
        if not p.exists():
            continue
        preds = {}
        for vname, v in views.items():
            preds[vname] = _predict_model(p, v)
            a, k, m = _agreement(preds[vname], d.y)
            rows.append({"sumber": name, "teks": vname, "setuju": round(a, 4), "kappa": round(k, 3)})
    if inset_dir and Path(inset_dir).exists():
        for vname, v in views.items():
            a, k, _ = _agreement(_inset_score(v, inset_dir), d.y)
            rows.append({"sumber": "InSet", "teks": vname, "setuju": round(a, 4), "kappa": round(k, 3)})
    base = float(d.y.astype(str).value_counts(normalize=True).iloc[0])
    r = pd.DataFrame(rows, columns=["sumber", "teks", "setuju", "kappa"]).sort_values("setuju", ascending=False).reset_index(drop=True)
    top = r.iloc[0] if len(r) else None
    if top is None:
        verdict = "Tidak ada calon yang bisa diuji."
    elif top.setuju >= .98:
        verdict = f"Agreement tinggi dengan {top.sumber}; belum membuktikan sumber label. Audit baseline mayoritas, mapping dan gold independen."
    elif top.setuju >= .85:
        verdict = f"Label dekat dengan {top.sumber}: mungkin turunan, voting, atau preprocessing lain. Coba varian."
    else:
        verdict = "Tidak ada sumber yang terbukti. Label belum tentu manual: sumbernya bisa model yang tidak kita punya."
    print(f"{len(d)} baris unik | kelas mayoritas {base:.3f}\n{verdict}")
    return r

# ===== jalankan =====
if CHECK_LABEL:
    print(label_fingerprint(train[TEXT], train[TARGET], MODELS, inset_dir=MODELS.parent / "data_ext/inset").to_string())
else:
    print("CHECK_LABEL=False: lewati. Nyalakan bila pretrained diizinkan.")


## Cleaning data

Kosong, duplikat, label bertentangan, dan grup untuk fold. Test tidak dibuang.

In [ ]:
import re
import numpy as np
import pandas as pd

_PSEUDO = {"", "nan", "none", "null", "-", "--", "n/a", "na", "[sticker]"}

def _groups_minhash(keys, threshold=.8, num_perm=64, shingle=5):
    from datasketch import MinHash, MinHashLSH
    lsh, par = MinHashLSH(threshold=threshold, num_perm=num_perm), list(range(len(keys)))
    def find(x):
        while par[x] != x:
            par[x] = par[par[x]]; x = par[x]
        return x
    for i, t in enumerate(keys):
        m = MinHash(num_perm=num_perm)
        for j in range(max(len(t) - shingle + 1, 1)):
            m.update(t[j:j + shingle].encode())
        for k in lsh.query(m):
            par[find(i)] = find(int(k))
        lsh.insert(str(i), m)
    return pd.factorize(np.array([find(i) for i in range(len(keys))]))[0]

def clean_dataset(df, text="text", target=None, dedup=True, near_dup=False, conflict=None, drop_empty=True, near_threshold=.8):
    """Kembalikan df bersih dengan kolom _group untuk fold. conflict: 'majority', 'drop', atau None."""
    n0 = len(df)
    df = df.copy()
    df[text] = df[text].fillna("").astype(str).str.strip()
    if drop_empty:
        df = df[~df[text].str.lower().isin(_PSEUDO)]
    key = df[text].str.lower().str.replace(r"\W+", " ", regex=True).str.strip()
    df["_group"] = pd.factorize(key)[0]
    if near_dup:
        df["_group"] = _groups_minhash(key.tolist(), near_threshold)
    n_conf = 0
    if target is not None and conflict:
        g = df.groupby(key)[target]
        bad = g.transform("nunique") > 1
        n_conf = int(bad.sum())
        if conflict == "majority":
            df[target] = g.transform(lambda s: s.mode().iloc[0])
        elif conflict == "drop":
            df, key = df[~bad], key[~bad]
    if dedup:
        # Konflik label tetap utuh dan satu grup; tidak memilih label pertama diam-diam.
        keep = ~pd.DataFrame({'key':key,'target':df[target]}).duplicated() if target is not None and conflict is None else ~key.duplicated()
        df = df[keep]
    print(f"baris {n0} -> {len(df)} | konflik label {n_conf} | grup {df._group.nunique()}")
    return df.reset_index(drop=True)

def clean_test(df, text="text"):
    """Test tidak dibuang: hanya isi kosong."""
    df = df.copy()
    df[text] = df[text].fillna("").astype(str)
    return df

# ===== jalankan =====
train = clean_dataset(train, TEXT, TARGET)      # near_dup=True untuk copypasta
test = clean_test(test, TEXT)
groups = train["_group"].to_numpy()


## Preprocessing

Satu fungsi, ganti preset di Setup, atau ubah satu langkah: preprocess_corpus(t, 'tfidf', emoji='remove').

In [ ]:
import html
import re
import unicodedata

SLANG = {"yg": "yang", "gk": "tidak", "ga": "tidak", "gak": "tidak", "nggak": "tidak", "tdk": "tidak", "bgt": "banget", "tp": "tapi", "klo": "kalau", "kalo": "kalau",
         "udh": "sudah", "dah": "sudah", "dpt": "dapat", "jg": "juga", "jd": "jadi", "bnyk": "banyak", "karna": "karena", "krn": "karena", "utk": "untuk", "jgn": "jangan",
         "org": "orang", "dr": "dari", "lgi": "lagi", "lg": "lagi", "bnr": "benar", "sdh": "sudah", "dgn": "dengan", "aja": "saja"}
STOPWORDS = set("""ada adalah agar akan aku anda antara apa apakah atau bagi bahwa banyak beberapa begitu bila bisa dalam dan dari dengan di dia dua hal hanya harus ia ini itu
jadi jika juga kami kamu karena ke kepada ketika kita lagi lain lebih maka masih mereka mungkin nya oleh pada para saat saja sama sampai sangat satu saya sebagai sebuah sedang
sehingga sekali selalu semua seperti setelah sudah supaya tapi telah tentang tersebut tetapi untuk yaitu yang""".split())   # sengaja tanpa tidak, bukan, jangan, belum, kurang
NEGATION = "tidak|tak|bukan|jangan|belum|kurang|tanpa"
_MOJIBAKE = re.compile("[\u00c3\u00c2\u00e2\u00f0]")

PRESETS = {
    # keep = biarkan, token = ganti penanda, remove = hapus
    "none":        dict(unicode=False, html=False, url="keep", email="keep", phone="keep", mention="keep", hashtag="keep", emoji="keep", number="keep", lower=False, repeat=False, punct="keep", slang=False, negation=False, stopword=False),
    "tfidf":       dict(unicode=True, html=True, url="token", email="token", phone="token", mention="token", hashtag="keep", emoji="keep", number="keep", lower=True, repeat=True, punct="keep", slang=False, negation=False, stopword=False),
    "social":      dict(unicode=True, html=True, url="token", email="token", phone="token", mention="token", hashtag="word", emoji="text", number="keep", lower=True, repeat=True, punct="keep", slang=True, negation=False, stopword=False),
    "topic":       dict(unicode=True, html=True, url="remove", email="remove", phone="remove", mention="remove", hashtag="word", emoji="remove", number="remove", lower=True, repeat=True, punct="remove", slang=True, negation=False, stopword=True),
    "transformer": dict(unicode=True, html=True, url="keep", email="keep", phone="keep", mention="keep", hashtag="keep", emoji="keep", number="keep", lower=False, repeat=False, punct="keep", slang=False, negation=False, stopword=False),
    "indobertweet": dict(unicode=True, html=True, url="HTTPURL", email="keep", phone="keep", mention="@USER", hashtag="keep", emoji="text", number="keep", lower=True, repeat=False, punct="keep", slang=False, negation=False, stopword=False),
}

def _sub(text, pattern, how, token):
    return text if how == "keep" else re.sub(pattern, " " if how == "remove" else f" {token if how == 'token' else how} ", text)

def preprocess(text, preset="tfidf", slang_dict=SLANG, stopwords=STOPWORDS, **override):
    """preprocess('Halo http://a.id', 'social'). Ubah satu langkah: preprocess(t, 'tfidf', emoji='remove')."""
    c = {**PRESETS[preset], **override}
    t = "" if text is None or text != text else str(text)
    if c["unicode"]:
        if _MOJIBAKE.search(t):                         # teks rusak encoding
            import ftfy
            t = ftfy.fix_text(t)
        t = unicodedata.normalize("NFKC", t).replace("​", "")
    if c["html"]:
        t = re.sub(r"<[^>]+>", " ", html.unescape(t))
    t = _sub(t, r"[\w.+-]+@[\w-]+\.[\w.-]+", c["email"], "EMAIL")          # email sebelum mention
    t = _sub(t, r"https?://\S+|www\.\S+", c["url"], "URL")
    t = _sub(t, r"(?<!\d)(?:\+?62|0)8\d{7,11}(?!\d)", c["phone"], "TELEPON")
    t = _sub(t, r"@\w+", c["mention"], "USER")
    if c["hashtag"] == "word":
        t = re.sub(r"#(\w+)", r" \1 ", t)
    elif c["hashtag"] != "keep":
        t = _sub(t, r"#\w+", c["hashtag"], "HASHTAG")
    if c["emoji"] != "keep":
        import emoji
        if c["emoji"] == "id":                          # emoji jadi kata Indonesia
            from indoNLP.preprocessing import emoji_to_words
            t = emoji_to_words(t, delimiter=(" ", " "))
        else:
            t = emoji.demojize(t, delimiters=(" ", " ")) if c["emoji"] == "text" else emoji.replace_emoji(t, " " if c["emoji"] == "remove" else " EMOJI ")
    t = _sub(t, r"\d+(?:[.,]\d+)*", c["number"], "ANGKA")
    if c["lower"]:
        keep = {m: m for m in ("HTTPURL", "@USER", "URL", "EMAIL", "TELEPON", "USER", "ANGKA", "EMOJI", "HASHTAG") if m in t}
        t = t.lower()
        for m in keep:
            t = re.sub(r"(?<!\w)" + re.escape(m.lower()) + r"(?!\w)", m, t)
    if c["repeat"]:
        t = re.sub(r"(.)\1{2,}", r"\1\1", t)
    if c["punct"] == "remove":
        t = re.sub(r"[^\w\s]", " ", t)
    if c["slang"]:
        t = re.sub(r"\b\w+\b", lambda m: slang_dict.get(m.group(0).lower(), m.group(0)), t)
    if c["negation"]:
        t = re.sub(rf"\b({NEGATION})\s+(\w+)", r"\1_\2", t, flags=re.I)
    if c["stopword"]:
        t = " ".join(w for w in t.split() if w.lower() not in stopwords)
    return re.sub(r"\s+", " ", t).strip()

def preprocess_corpus(texts, preset="tfidf", stem=False, slang_csv=None, **override):
    """Versi cepat: teks identik hanya diproses sekali, stem per kata unik. Kembalikan list."""
    slang = dict(SLANG)
    if slang_csv:
        import pandas as pd
        lex = pd.read_csv(slang_csv)
        slang.update(dict(zip(lex["slang"].astype(str), lex["formal"].astype(str))))
    texts = list(texts)
    cache = {t: preprocess(t, preset, slang_dict=slang, **override) for t in {x if x == x else None for x in texts}}
    out = [cache[t if t == t else None] for t in texts]
    if stem:
        from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
        st = StemmerFactory().create_stemmer()
        m = {w: st.stem(w) for w in {w for d in out for w in d.split()}}
        out = [" ".join(m[w] for w in d.split()) for d in out]
    return out

def text_versions(df, text="text", slang_csv=None):
    """Tiga versi agar efek normalisasi bisa diuji: text_raw, text_clean (makna utuh), text_norm (slang dinormalkan)."""
    d = df.copy()
    d["text_raw"] = d[text]
    d["text_clean"] = preprocess_corpus(d[text], "tfidf")
    d["text_norm"] = preprocess_corpus(d[text], "social", slang_csv=slang_csv)
    return d

# ===== jalankan =====
texts = preprocess_corpus(train[TEXT], PREPROCESS)
test_texts = preprocess_corpus(test[TEXT], PREPROCESS)
for a, b in list(zip(train[TEXT], texts))[:3]:
    print(a[:80], "->", b[:80])


## Fitur

Nama vectorizer dipakai di EXPERIMENTS. Cek ukuran matriks di sini.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, HashingVectorizer, TfidfVectorizer
from sklearn.pipeline import make_pipeline, make_union
from sklearn.preprocessing import FunctionTransformer, Normalizer

def _tokens(t):
    return re.findall(r"\w+", str(t).lower())

def _meta(texts):
    s = pd.Series(list(texts)).fillna("").astype(str)
    n = s.str.len().clip(lower=1)
    return sparse.csr_matrix(pd.DataFrame({"len": np.log1p(n), "words": np.log1p(s.str.split().str.len()), "upper": s.str.count(r"[A-Z]") / n, "digit": s.str.count(r"\d") / n,
                                           "punct": s.str.count(r"[^\w\s]") / n, "excl": s.str.count("!").clip(upper=5), "url": s.str.count(r"https?://|URL"), "mention": s.str.count(r"@\w+|USER")}).to_numpy("float32"))

class RawText(BaseEstimator, TransformerMixin):
    """Teks mentah untuk model neural."""
    def fit(self, X, y=None): return self
    def transform(self, X): return list(X)

class GensimVec(BaseEstimator, TransformerMixin):
    """kind: w2v, fasttext, doc2vec. idf=True: rata-rata berbobot."""
    def __init__(self, kind="w2v", dim=200, window=5, min_count=2, epochs=5, idf=False, workers=4, seed=42):
        self.kind, self.dim, self.window, self.min_count, self.epochs, self.idf, self.workers, self.seed = kind, dim, window, min_count, epochs, idf, workers, seed
    def fit(self, X, y=None):
        from gensim.models import FastText, Word2Vec
        from gensim.models.doc2vec import Doc2Vec, TaggedDocument
        toks = [_tokens(t) for t in X]
        kw = dict(vector_size=self.dim, min_count=self.min_count, epochs=self.epochs, workers=self.workers, seed=self.seed)
        if self.kind == "doc2vec":
            self.m_ = Doc2Vec([TaggedDocument(t, [i]) for i, t in enumerate(toks)], dm=0, **kw)
        else:
            self.m_ = (FastText if self.kind == "fasttext" else Word2Vec)(toks, window=self.window, **kw).wv
        self.w_ = None
        if self.idf:
            t = TfidfVectorizer(token_pattern=r"\w+").fit(X)
            self.w_ = dict(zip(t.get_feature_names_out(), t.idf_))
        return self
    def transform(self, X):
        out = np.zeros((len(X), self.dim), dtype="float32")
        for i, t in enumerate(X):
            w = _tokens(t)
            if self.kind == "doc2vec":
                out[i] = self.m_.infer_vector(w)
                continue
            w = w if self.kind == "fasttext" else [x for x in w if x in self.m_]
            if w:
                a = np.array([self.w_.get(x, 1.0) for x in w])[:, None] if self.w_ else np.ones((len(w), 1))
                out[i] = (self.m_[w] * a).sum(0) / a.sum()
        return out

class EmbedVec(BaseEstimator, TransformerMixin):
    """Embedding SentenceTransformer dengan cache per teks agar tiap fold tidak encode ulang."""
    _cache = {}
    def __init__(self, model_path=None, prefix="", batch=128):
        self.model_path, self.prefix, self.batch = model_path, prefix, batch
    def fit(self, X, y=None): return self
    def transform(self, X):
        c = EmbedVec._cache.setdefault(str(self.model_path), {})
        miss = [t for t in dict.fromkeys(map(str, X)) if t not in c]
        if miss:
            from sentence_transformers import SentenceTransformer
            m = SentenceTransformer(str(self.model_path), local_files_only=True)
            for t, v in zip(miss, m.encode([self.prefix + t for t in miss], batch_size=self.batch, normalize_embeddings=True, show_progress_bar=False)):
                c[t] = v.astype("float32")
        return np.vstack([c[str(t)] for t in X])

def make_vectorizer(kind="tfidf_union", **kw):
    """Semua hasilnya punya fit_transform dan transform."""
    word = dict(ngram_range=(1, 2), min_df=2, max_df=.95, sublinear_tf=True, dtype="float32", token_pattern=r"(?u)\b\w+\b")
    char = dict(analyzer="char_wb", ngram_range=(2, 5), min_df=3, max_features=500_000, sublinear_tf=True, dtype="float32")
    if kind == "tfidf_word": return TfidfVectorizer(**{**word, **kw})
    if kind == "tfidf_char": return TfidfVectorizer(**{**char, **kw})
    if kind == "tfidf_union": return make_union(TfidfVectorizer(**word), TfidfVectorizer(**char))
    if kind == "count": return CountVectorizer(**{**dict(ngram_range=(1, 1), min_df=2, max_df=.95), **kw})
    if kind == "binary": return CountVectorizer(**{**dict(ngram_range=(1, 2), min_df=2, binary=True), **kw})
    if kind == "hashing": return HashingVectorizer(**{**dict(n_features=2 ** 20, alternate_sign=False, ngram_range=(1, 2), dtype="float32"), **kw})
    if kind == "lsa": return make_pipeline(TfidfVectorizer(**word), TruncatedSVD(kw.pop("k", 256), random_state=42), Normalizer(copy=False))
    if kind == "meta": return FunctionTransformer(_meta, validate=False)
    if kind == "tfidf_meta": return make_union(TfidfVectorizer(**word), FunctionTransformer(_meta, validate=False))
    if kind in ("w2v", "fasttext", "doc2vec"): return GensimVec(kind, **kw)
    if kind == "w2v_idf": return GensimVec("w2v", idf=True, **kw)
    if kind == "embed": return EmbedVec(**kw)
    if kind == "raw": return RawText()
    raise ValueError(f"vectorizer tidak dikenal: {kind}")

# ===== jalankan =====
for v in dict.fromkeys(e[0] for e in EXPERIMENTS):
    print(v, make_vectorizer(v).fit_transform(texts[:5000]).shape)


## Validasi

Encode label dan buat fold. Grup mencegah teks sama melintasi fold. Split waktu hanya memvalidasi sebagian baris (COV).

In [ ]:
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd

def encode_labels(y):
    """Kembalikan (y_int, kelas). Decode: kelas[pred]."""
    classes, yi = np.unique(np.asarray(y).astype(str), return_inverse=True)
    return yi, classes

def write_submission(sample_path, preds, out="submission.csv", pred_col=None, id_col=None, allowed=None):
    """Salin struktur contoh submission, isi prediksi, validasi, dan cetak hash."""
    s = pd.read_csv(sample_path, dtype=str)
    id_col = id_col or s.columns[0]
    pred_col = pred_col or s.columns[-1]
    assert len(s) == len(preds), f"jumlah baris {len(preds)} != contoh {len(s)}"
    s[pred_col] = preds
    assert s[pred_col].notna().all(), "ada NaN"
    assert s[id_col].is_unique, "id ganda"
    if allowed is not None:
        bad = set(s[pred_col]) - set(allowed)
        assert not bad, f"nilai asing: {bad}"
    s.to_csv(out, index=False)
    print(out, s.shape, "sha256", hashlib.sha256(Path(out).read_bytes()).hexdigest()[:16], s[pred_col].value_counts(normalize=True).head(5).round(3).to_dict())
    return s

import re
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, KFold, LeaveOneGroupOut, StratifiedGroupKFold, StratifiedKFold

def _ml_folds(Y, n, seed):
    raw = np.asarray(Y, dtype=float)
    assert np.isin(raw[np.isfinite(raw)], [0, 1]).all(), "Target harus 0/1/NaN"
    Y, rng = np.hstack([raw == 1, np.isfinite(raw)]), np.random.default_rng(seed)
    left, fold = np.ones(len(Y), bool), -np.ones(len(Y), int)
    need, cap = np.tile(Y.sum(0) / n, (n, 1)).astype(float), np.full(n, len(Y) / n)
    while left.any():
        c = Y[left].sum(0)
        c[c == 0] = 10 ** 9
        lab = int(c.argmin())
        pool = np.where(left)[0] if c[lab] == 10 ** 9 else rng.permutation(np.where(left & Y[:, lab])[0])
        for i in pool:
            d = need[:, lab]
            cand = np.where(d == d.max())[0]
            f = int(cand[cap[cand].argmax()])
            fold[i], left[i] = f, False
            cap[f] -= 1
            need[f, Y[i]] -= 1
    return [(np.where(fold != k)[0], np.where(fold == k)[0]) for k in range(n)]

def make_folds(y=None, groups=None, times=None, kind="auto", n=5, seed=42, valid_share=.2):
    """Kembalikan list (idx_train, idx_valid). time: satu holdout terbaru."""
    if kind == "auto":
        y_ = None if y is None else np.asarray(y)
        if times is not None: kind = "time"
        elif y_ is None: kind = "group" if groups is not None else "kfold"
        elif y_.ndim == 2: kind = "multilabel"
        elif np.issubdtype(y_.dtype, np.floating) and len(np.unique(y_)) > 20: kind = "regression" if groups is None else "group"
        else: kind = "strat_group" if groups is not None else "stratified"
        print("skema fold:", kind)
    if kind == "time":
        t = pd.to_datetime(pd.Series(list(times)), errors="coerce")
        cut = t.quantile(1 - valid_share)
        return [(np.where(t < cut)[0], np.where(t >= cut)[0])]
    if kind == "source": return list(LeaveOneGroupOut().split(np.zeros(len(groups)), groups=groups))
    N = len(y) if y is not None else len(groups)
    Z = np.zeros(N)
    if kind == "multilabel":
        if groups is None: return _ml_folds(y, n, seed)
        gid, inverse = np.unique(groups, return_inverse=True)
        raw = np.asarray(y, float)
        profile = np.array([np.r_[np.any(raw[inverse==i]==1,axis=0), np.any(np.isfinite(raw[inverse==i]),axis=0)] for i in range(len(gid))])
        return [(np.flatnonzero(np.isin(inverse,a)),np.flatnonzero(np.isin(inverse,b))) for a,b in _ml_folds(profile,n,seed)]
    if kind == "kfold": return list(KFold(n, shuffle=True, random_state=seed).split(Z))
    if kind == "group": return list(GroupKFold(n, shuffle=True, random_state=seed).split(Z, groups=groups))
    if kind == "strat_group": return list(StratifiedGroupKFold(n, shuffle=True, random_state=seed).split(Z, y, groups))
    if kind == "regression":
        v = np.asarray(y, float)
        return list(StratifiedKFold(n, shuffle=True, random_state=seed).split(Z, np.digitize(v, np.unique(np.quantile(v, np.linspace(0, 1, 11)[1:-1])))))
    if kind == "stratified": return list(StratifiedKFold(n, shuffle=True, random_state=seed).split(Z, y))
    raise ValueError(kind)

def check_folds(texts, folds):
    """Persen teks validasi yang juga ada di latih. Harus mendekati 0 kecuali test memang overlap."""
    k = np.array([re.sub(r"\W+", " ", str(t).lower()).strip() for t in texts], dtype=object)
    r = [round(float(np.isin(k[b], k[a]).mean()), 4) for a, b in folds]
    print("overlap per fold:", r)
    return r

def _proba(m, X):
    if hasattr(m, "predict_proba"):
        return m.predict_proba(X)
    if hasattr(m, "decision_function"):
        raise ValueError('Margin bukan probabilitas. Gunakan predict untuk F1 atau classifier terkalibrasi untuk probability ensemble.')
    return np.asarray(m.predict(X)).reshape(-1, 1)

def oof_predict(make_vec, make_mdl, texts, y, folds, test_texts=None, sample_weight=None):
    """make_vec, make_mdl: fungsi tanpa argumen. Vectorizer di-fit di dalam fold. Kembalikan (oof, test rata-rata fold)."""
    texts, y = np.asarray(texts, dtype=object), np.asarray(y)
    oof, test = None, None
    for tr, va in folds:
        vec, model = make_vec(), make_mdl()
        Xtr, Xva = vec.fit_transform(list(texts[tr]), y[tr]), vec.transform(list(texts[va]))
        kw = {} if sample_weight is None else {"sample_weight": np.asarray(sample_weight)[tr]}
        model.fit(Xtr, y[tr], **kw)
        p = _proba(model, Xva)
        classes = np.unique(y) if y.ndim == 1 and hasattr(model, 'classes_') else None
        if classes is not None:
            aligned = np.zeros((len(va),len(classes)))
            aligned[:,np.searchsorted(classes,model.classes_)] = p
            p = aligned
        if oof is None:
            oof = np.full((len(y), p.shape[1]), np.nan, dtype="float32")
        oof[va] = p
        if test_texts is not None:
            t = _proba(model, vec.transform(list(test_texts)))
            if classes is not None:
                aligned = np.zeros((len(t),len(classes)))
                aligned[:,np.searchsorted(classes,model.classes_)] = t
                t = aligned
            test = t / len(folds) if test is None else test + t / len(folds)
    return oof, test

# ===== jalankan =====
y, classes = encode_labels(train[TARGET])
folds = make_folds(y, groups if FOLD_KIND != "time" else None, None if TIME is None else train[TIME], kind=FOLD_KIND, n=N_FOLDS, seed=SEED)
check_folds(texts, folds)
COV = np.unique(np.concatenate([b for _, b in folds]))   # baris yang punya OOF: split waktu hanya memvalidasi sebagian
print("kelas", list(classes), "| fold", [len(b) for _, b in folds], "| baris tervalidasi", len(COV), "dari", len(y))
clock("validasi")


## Model

Melatih semua di EXPERIMENTS dengan fold yang sama, lalu memilih yang terbaik.

In [ ]:
import re
from collections import Counter
from functools import partial
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin

def _net(kind, vocab, n_out, dim=200, hidden=128, drop=.3):
    import torch
    import torch.nn as nn
    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            self.emb, self.drop = nn.Embedding(vocab, dim, padding_idx=0), nn.Dropout(drop)
            if kind == "cnn":
                self.conv = nn.ModuleList(nn.Conv1d(dim, hidden, k, padding=k // 2) for k in (3, 4, 5))
                self.out = nn.Linear(hidden * 3, n_out)
            else:
                bi = kind in ("bigru", "bilstm", "attn")
                self.rnn = (nn.LSTM if "lstm" in kind else nn.GRU)(dim, hidden, batch_first=True, bidirectional=bi)
                self.att = nn.Linear(hidden * (2 if bi else 1), 1)
                self.out = nn.Linear(hidden * (2 if bi else 1), n_out)
        def forward(self, x):
            e = self.emb(x)
            if kind == "cnn":
                return self.out(self.drop(torch.cat([c(e.transpose(1, 2)).relu().amax(2) for c in self.conv], 1)))
            h, _ = self.rnn(e)
            w = self.att(h).squeeze(-1).masked_fill(x == 0, -1e4).softmax(1) if kind == "attn" else (x != 0).float() / (x != 0).sum(1, keepdim=True).clamp(min=1)
            return self.out(self.drop((h * w.unsqueeze(-1)).sum(1)))
    return Net()

class NeuralText(BaseEstimator, ClassifierMixin):
    """Terima list teks. Pakai bersama make_vectorizer('raw')."""
    def __init__(self, kind="cnn", epochs=3, batch=128, lr=1e-3, max_len=128, min_count=2, device=None, seed=42):
        self.kind, self.epochs, self.batch, self.lr, self.max_len, self.min_count, self.device, self.seed = kind, epochs, batch, lr, max_len, min_count, device, seed
    def _ids(self, T):
        a = np.zeros((len(T), self.max_len), dtype="int64")
        for i, t in enumerate(T):
            w = [self.v_.get(x, 1) for x in re.findall(r"\w+", str(t).lower())[:self.max_len]]
            a[i, :len(w)] = w
        return a
    def fit(self, T, y):
        import torch
        torch.manual_seed(self.seed)
        self.d_ = self.device or ("cuda" if torch.cuda.is_available() else "cpu")
        c = Counter(w for t in T for w in re.findall(r"\w+", str(t).lower()))
        self.v_ = {w: i + 2 for i, (w, k) in enumerate(c.most_common()) if k >= self.min_count}
        self.classes_, yy = np.unique(y, return_inverse=True)
        X, yy = torch.tensor(self._ids(T)), torch.tensor(yy)
        self.n_ = _net(self.kind, len(self.v_) + 2, len(self.classes_)).to(self.d_)
        opt, loss = torch.optim.AdamW(self.n_.parameters(), lr=self.lr), torch.nn.CrossEntropyLoss()
        for _ in range(self.epochs):
            self.n_.train()
            for b in torch.randperm(len(yy)).split(self.batch):
                opt.zero_grad(); loss(self.n_(X[b].to(self.d_)), yy[b].to(self.d_)).backward(); opt.step()
        return self
    def predict_proba(self, T):
        import torch
        self.n_.eval()
        with torch.inference_mode():
            return np.vstack([self.n_(b.to(self.d_)).softmax(1).float().cpu().numpy() for b in torch.tensor(self._ids(T)).split(512)])
    def predict(self, T):
        return self.classes_[self.predict_proba(T).argmax(1)]

def make_model(name="lr", seed=42, n_jobs=8, **kw):
    if name in ("cnn", "gru", "bigru", "lstm", "bilstm", "attn"):
        return NeuralText(name, seed=seed, **kw)
    from sklearn.linear_model import LogisticRegression, Ridge, RidgeClassifier, SGDClassifier
    if name == "lr": return LogisticRegression(**{**dict(C=4.0, max_iter=2000), **kw})
    if name == "sgd": return SGDClassifier(**{**dict(loss="log_loss", alpha=1e-6, random_state=seed), **kw})
    if name == "ridge_clf": return RidgeClassifier(**kw)
    if name in ("svc", "svc_cal"):
        from sklearn.svm import LinearSVC
        m = LinearSVC(**{**dict(C=.5, dual="auto", max_iter=5000), **kw})
        if name == "svc":
            return m
        from sklearn.calibration import CalibratedClassifierCV
        return CalibratedClassifierCV(m, method="sigmoid", cv=3)
    if name in ("cnb", "mnb"):
        from sklearn.naive_bayes import ComplementNB, MultinomialNB
        return (ComplementNB if name == "cnb" else MultinomialNB)(**{**dict(alpha=.3), **kw})
    if name in ("lgbm", "lgbm_reg", "lgbm_l1"):
        import lightgbm as lgb
        p = {**dict(n_estimators=500, learning_rate=.05, num_leaves=63, subsample=.8, subsample_freq=1, colsample_bytree=.5, random_state=seed, n_jobs=n_jobs, verbose=-1), **kw}
        return lgb.LGBMClassifier(**p) if name == "lgbm" else lgb.LGBMRegressor(objective="l1" if name == "lgbm_l1" else "l2", **p)
    if name == "xgb":
        from xgboost import XGBClassifier
        return XGBClassifier(**{**dict(n_estimators=500, learning_rate=.05, max_depth=6, tree_method="hist", subsample=.8, colsample_bytree=.5, random_state=seed), **kw})
    if name in ("rf", "rf_reg"):
        from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
        return (RandomForestClassifier if name == "rf" else RandomForestRegressor)(**{**dict(n_estimators=300, min_samples_leaf=2, n_jobs=n_jobs, random_state=seed), **kw})
    if name == "mlp":
        from sklearn.neural_network import MLPClassifier
        return MLPClassifier(**{**dict(hidden_layer_sizes=(256,), early_stopping=True, max_iter=100, random_state=seed), **kw})
    if name == "smote_lr":
        from imblearn.over_sampling import SMOTE
        from imblearn.pipeline import make_pipeline as imb
        return imb(SMOTE(k_neighbors=kw.get("k", 5), random_state=seed), LogisticRegression(max_iter=2000))
    if name in ("ovr", "chain"):
        base = LogisticRegression(**{**dict(C=4.0, max_iter=2000), **kw})
        if name == "ovr":
            from sklearn.multiclass import OneVsRestClassifier
            return OneVsRestClassifier(base, n_jobs=4)
        from sklearn.multioutput import ClassifierChain
        return ClassifierChain(base, order="random", random_state=seed)
    if name in ("ridge", "ridge_log"):
        m = Ridge(**{**dict(alpha=1.0, solver="sag", random_state=seed), **kw})
        if name == "ridge":
            return m
        from sklearn.compose import TransformedTargetRegressor
        return TransformedTargetRegressor(m, func=np.log1p, inverse_func=np.expm1)
    if name in ("dummy", "dummy_reg"):
        from sklearn.dummy import DummyClassifier, DummyRegressor
        return DummyClassifier(strategy="prior") if name == "dummy" else DummyRegressor(strategy=kw.get("strategy", "mean"))
    raise ValueError(f"model tidak dikenal: {name}")

import numpy as np
from sklearn.metrics import accuracy_score, average_precision_score, f1_score, hamming_loss, log_loss, precision_score, recall_score, roc_auc_score, top_k_accuracy_score

def cls_report(y, proba=None, pred=None):
    """proba: (n, k). pred otomatis argmax bila kosong."""
    y = np.asarray(y)
    pred = pred if pred is not None else np.asarray(proba).argmax(1)
    r = dict(acc=accuracy_score(y, pred), macro_f1=f1_score(y, pred, average="macro", zero_division=0), weighted_f1=f1_score(y, pred, average="weighted", zero_division=0),
             precision=precision_score(y, pred, average="macro", zero_division=0), recall=recall_score(y, pred, average="macro", zero_division=0))
    if proba is not None:
        p = np.asarray(proba)
        r["log_loss"] = log_loss(y, np.clip(p, 1e-6, 1 - 1e-6), labels=np.arange(p.shape[1]))
        if p.shape[1] == 2:
            r.update(f1_pos=f1_score(y, pred, zero_division=0), roc_auc=roc_auc_score(y, p[:, 1]), pr_auc=average_precision_score(y, p[:, 1]))
        else:
            r["roc_auc_ovr"] = roc_auc_score(y, p, multi_class="ovr", labels=np.arange(p.shape[1]))
            r["top2"] = top_k_accuracy_score(y, p, k=2, labels=np.arange(p.shape[1]))
    return {k: round(float(v), 4) for k, v in r.items()}

def multilabel_report(Y, pred):
    Y, pred = np.asarray(Y), np.asarray(pred)
    return {k: round(float(v), 4) for k, v in dict(micro_f1=f1_score(Y, pred, average="micro", zero_division=0), macro_f1=f1_score(Y, pred, average="macro", zero_division=0),
            samples_f1=f1_score(Y, pred, average="samples", zero_division=0), hamming=hamming_loss(Y, pred), subset_acc=accuracy_score(Y, pred)).items()}

import json
import time
from pathlib import Path
import numpy as np
import pandas as pd

def save_run(name, oof=None, test=None, score=None, config=None, ids=None, folder="runs"):
    d = Path(folder) / name
    d.mkdir(parents=True, exist_ok=True)
    if oof is not None:
        np.save(d / "oof.npy", oof)
    if test is not None:
        np.save(d / "test.npy", test)
    if ids is not None:
        np.save(d / "ids.npy", np.asarray(ids))
    (d / "meta.json").write_text(json.dumps({"score": score, "config": config, "time": time.strftime("%H:%M:%S")}, default=str, indent=1))
    with open(Path(folder) / "log.csv", "a", encoding="utf-8") as f:
        f.write(f"{time.strftime('%H:%M:%S')},{name},{score}\n")

def load_run(name, folder="runs"):
    d = Path(folder) / name
    return {p.stem: np.load(p, allow_pickle=True) for p in d.glob("*.npy")} | json.loads((d / "meta.json").read_text())

def leaderboard(folder="runs"):
    return pd.read_csv(Path(folder) / "log.csv", names=["waktu", "run", "skor"]).sort_values("skor", ascending=False)

# ===== jalankan =====
from sklearn.metrics import f1_score
macro = lambda a, b: f1_score(a, b, average="macro", zero_division=0)
results = {}
print("baseline", cls_report(y[COV], oof_predict(lambda: make_vectorizer("count"), lambda: make_model("dummy"), texts, y, folds)[0][COV]))
for v, m in EXPERIMENTS:
    oof, pred = oof_predict(lambda: make_vectorizer(v), lambda: make_model(m), texts, y, folds, test_texts)
    results[f"{v}+{m}"] = (oof, pred)
    save_run(f"{v}+{m}", oof, pred, macro(y[COV], oof[COV].argmax(1)))
    print(f"{v}+{m}", cls_report(y[COV], oof[COV]))
best = max(results, key=lambda k: macro(y[COV], results[k][0][COV].argmax(1)))
oof, pred = results[best]
print("terbaik:", best)
clock("baseline")


## Transformer

Lewati bila USE_BERT=False. Satu holdout, bukan 5 fold. Hasil TF-IDF tetap jadi fallback.

In [ ]:
from pathlib import Path
import numpy as np
import torch
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          EarlyStoppingCallback, Trainer, TrainingArguments)

class TextDS(torch.utils.data.Dataset):
    def __init__(self, enc, labels=None):
        self.enc, self.labels = enc, labels
    def __len__(self):
        return len(self.enc["input_ids"])
    def __getitem__(self, i):
        d = {k: v[i] for k, v in self.enc.items()}
        if self.labels is not None:
            d["labels"] = self.labels[i]
        return d

class LossTrainer(Trainer):
    """loss_fn(logits, labels) opsional, misalnya focal loss."""
    def __init__(self, *a, loss_fn=None, **k):
        super().__init__(*a, **k)
        self.loss_fn = loss_fn
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        if self.loss_fn is None:
            return super().compute_loss(model, inputs, return_outputs=return_outputs, **kw)
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = self.loss_fn(out.logits, labels)
        return (loss, out) if return_outputs else loss

def train_transformer(model_path, texts, y, valid_texts=None, valid_y=None, texts2=None, valid_texts2=None, problem="single",
                      num_labels=None, max_len=128, batch=16, accum=1, epochs=1, lr=2e-5, warmup=.1, eval_every=500, patience=3,
                      out="run_tf", seed=42, loss_fn=None, smoothing=0.0, resume=False, fp16=False):
    tok = AutoTokenizer.from_pretrained(str(model_path), local_files_only=True)
    def enc(a, b):
        return tok([str(t) for t in a], None if b is None else [str(t) for t in b], truncation=True, max_length=max_len)
    def lab(v):
        v = np.asarray(v)
        return v.astype("float32").tolist() if problem != "single" else v.astype("int64").tolist()
    n = num_labels or (1 if problem == "reg" else np.asarray(y).shape[1] if problem == "multi" else int(np.max(y)) + 1)
    ptype = {"single": "single_label_classification", "multi": "multi_label_classification", "reg": "regression"}[problem]
    model = AutoModelForSequenceClassification.from_pretrained(str(model_path), local_files_only=True, num_labels=n, problem_type=ptype, ignore_mismatched_sizes=True)
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported() and not fp16
    has_val = valid_texts is not None
    args = TrainingArguments(output_dir=out, per_device_train_batch_size=batch, per_device_eval_batch_size=batch * 2, gradient_accumulation_steps=accum,
                             num_train_epochs=epochs, learning_rate=lr, warmup_steps=warmup, label_smoothing_factor=smoothing,
                             eval_strategy="steps" if has_val else "no", eval_steps=eval_every, save_strategy="steps" if has_val else "no", save_steps=eval_every,
                             save_total_limit=2, load_best_model_at_end=has_val, metric_for_best_model="eval_loss", greater_is_better=False,
                             train_sampling_strategy="group_by_length", bf16=use_bf16, fp16=fp16 and torch.cuda.is_available(),
                             dataloader_num_workers=0, report_to="none", logging_steps=100, seed=seed, disable_tqdm=True)
    tr = LossTrainer(model=model, args=args, train_dataset=TextDS(enc(texts, texts2), lab(y)),
                     eval_dataset=TextDS(enc(valid_texts, valid_texts2), lab(valid_y)) if has_val else None,
                     processing_class=tok, data_collator=DataCollatorWithPadding(tok), loss_fn=loss_fn,
                     callbacks=[EarlyStoppingCallback(patience)] if has_val else None)
    if loss_fn is not None:
        tr.model_accepts_loss_kwargs = False
    from transformers.trainer_utils import get_last_checkpoint
    last = get_last_checkpoint(out) if resume and Path(out).is_dir() else None
    if last:
        print("lanjut dari", last)
    tr.train(resume_from_checkpoint=last)
    return tr.model, tok

@torch.inference_mode()
def predict_transformer(model, tok, texts, texts2=None, problem="single", max_len=128, batch=64):
    """Terurut panjang untuk hemat padding. single: softmax, multi: sigmoid, reg: nilai mentah."""
    dev = next(model.parameters()).device
    model.eval()
    texts = [str(t) for t in texts]
    order = np.argsort([len(t) for t in texts])
    out = None
    for i in range(0, len(texts), batch):
        idx = order[i:i + batch]
        a, b = [texts[j] for j in idx], None if texts2 is None else [str(texts2[j]) for j in idx]
        e = tok(a, b, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(dev)
        lg = model(**e).logits
        p = lg.softmax(1) if problem == "single" else lg.sigmoid() if problem == "multi" else lg
        p = p.float().cpu().numpy()
        if out is None:
            out = np.zeros((len(texts), p.shape[1]), dtype="float32")
        out[idx] = p
    return out

import numpy as np
from sklearn.model_selection import train_test_split

def subsample_idx(y, n, seed=42, groups=None):
    """Indeks n baris, proporsi kelas terjaga."""
    y = np.asarray(y)
    if n >= len(y):
        return np.arange(len(y))
    _, c = np.unique(y, return_counts=True)
    strat = y if c.min() >= 2 else None
    return train_test_split(np.arange(len(y)), train_size=n, random_state=seed, stratify=strat)[0]

# ===== jalankan =====
bert_valid, bert_test = None, None
if USE_BERT:
    bt, bt_test = preprocess_corpus(train[TEXT], "indobertweet"), preprocess_corpus(test[TEXT], "indobertweet")
    tr, va = folds[0]
    tr = tr[subsample_idx(y[tr], BERT_ROWS, SEED)]
    model, tok = train_transformer(BERT_MODEL, [bt[i] for i in tr], y[tr], [bt[i] for i in va[:8000]], y[va[:8000]], num_labels=len(classes), max_len=MAX_LEN, batch=BATCH, resume=RESUME)
    bert_valid = (va, predict_transformer(model, tok, [bt[i] for i in va], max_len=MAX_LEN))
    bert_test = predict_transformer(model, tok, bt_test, max_len=MAX_LEN)
    full = np.full((len(y), bert_valid[1].shape[1]), np.nan, dtype="float32"); full[va] = bert_valid[1]
    save_run("bert", full, bert_test, macro(y[va], bert_valid[1].argmax(1)))   # aman walau kernel mati setelah ini
    print("BERT holdout", cls_report(y[va], bert_valid[1]), "\nTF-IDF holdout", cls_report(y[va], oof[va]))
else:
    print("USE_BERT=False: lanjut dengan TF-IDF")


## Evaluasi & error analysis

Noise floor menentukan batas terima. Baca kesalahan sebelum menambah model.

In [ ]:
import numpy as np

def _sample(n, groups, rng):
    if groups is None:
        return rng.integers(0, n, n)
    g = np.asarray(groups)
    u = np.unique(g)
    pick = rng.choice(u, len(u))
    pos = {k: np.where(g == k)[0] for k in u}
    return np.concatenate([pos[k] for k in pick])

def noise_floor(y, pred, score_fn, n_boot=300, groups=None, seed=42):
    """Kembalikan (skor, SE, batas 2,5%, batas 97,5%). groups untuk cluster bootstrap."""
    y, pred, rng = np.asarray(y), np.asarray(pred), np.random.default_rng(seed)
    s = [score_fn(y[i], pred[i]) for i in (_sample(len(y), groups, rng) for _ in range(n_boot))]
    return round(score_fn(y, pred), 4), round(float(np.std(s)), 4), round(float(np.percentile(s, 2.5)), 4), round(float(np.percentile(s, 97.5)), 4)

def paired_gain(y, pred_a, pred_b, score_fn, n_boot=300, groups=None, seed=42):
    """B dibanding A pada baris yang sama. Terima B bila gain > 2*se."""
    y, a, b, rng = np.asarray(y), np.asarray(pred_a), np.asarray(pred_b), np.random.default_rng(seed)
    d = []
    for _ in range(n_boot):
        i = _sample(len(y), groups, rng)
        d.append(score_fn(y[i], b[i]) - score_fn(y[i], a[i]))
    return dict(gain=round(score_fn(y, b) - score_fn(y, a), 4), se=round(float(np.std(d)), 4), p_b_lebih_baik=round(float(np.mean(np.array(d) > 0)), 3))

def learning_curve_oof(fit_score, n_rows, shares=(.1, .25, .5, 1.0), seed=42):
    """fit_score(idx) melatih pada idx dan mengembalikan skor validasi tetap."""
    order = np.random.default_rng(seed).permutation(n_rows)
    return {s: fit_score(order[:int(n_rows * s)]) for s in shares}

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score

def worst_errors(texts, y, proba, k=30, names=None):
    y, p = np.asarray(y), np.asarray(proba)
    pred = p.argmax(1)
    d = pd.DataFrame({"text": list(texts), "y": y, "pred": pred, "conf": p.max(1)})[y != pred]
    if names is not None:
        d["y"], d["pred"] = d.y.map(dict(enumerate(names))), d.pred.map(dict(enumerate(names)))
    return d.sort_values("conf", ascending=False).head(k)

def random_errors(texts, y, pred, k=50, seed=42):
    d = pd.DataFrame({"text": list(texts), "y": np.asarray(y), "pred": np.asarray(pred)})
    d = d[d.y != d.pred]
    return d.sample(min(k, len(d)), random_state=seed).assign(kategori="")

def confused_pairs(y, pred, names=None, k=10):
    c = confusion_matrix(y, pred, normalize="true")
    np.fill_diagonal(c, 0)
    o = np.dstack(np.unravel_index(np.argsort(-c.ravel()), c.shape))[0][:k]
    return [((names[a] if names is not None else int(a)), (names[b] if names is not None else int(b)), round(float(c[a, b]), 3)) for a, b in o]

def per_label(y, pred, names=None):
    return pd.DataFrame(classification_report(y, pred, target_names=names, zero_division=0, output_dict=True)).T.round(3)

def segment_scores(y, pred, segments, score=lambda a, b: f1_score(a, b, average="macro", zero_division=0), min_n=30):
    d = pd.DataFrame({"y": np.asarray(y), "p": np.asarray(pred), "s": np.asarray(segments)})
    r = {k: (len(g), round(score(g.y, g.p), 4)) for k, g in d.groupby("s") if len(g) >= min_n}
    return pd.DataFrame(r, index=["n", "skor"]).T.sort_values("skor")

def residual_table(y, pred, bins=5):
    d = pd.DataFrame({"y": np.asarray(y, float), "p": np.asarray(pred, float)})
    d["err"] = d.p - d.y
    return d.groupby(pd.qcut(d.y, bins, duplicates="drop"), observed=True).err.agg(["count", "mean", "std"]).round(3)

# ===== jalankan =====
nf = COV[sample_idx(len(COV), 100000, SEED)]        # bootstrap pada sampel: data penuh terlalu lambat
print("skor, SE, batas bawah, batas atas:", noise_floor(y[nf], oof[nf].argmax(1), macro, 200, groups=None))
print(per_label(y[COV], oof[COV].argmax(1), list(classes)).to_string())
print("tertukar:", confused_pairs(y[COV], oof[COV].argmax(1), list(classes), 5))
print(worst_errors(train[TEXT].iloc[COV].tolist(), y[COV], oof[COV], 10, classes).to_string())


## Post-processing

Blend bila gain melebihi 2 SE, lalu threshold atau pengali dari OOF.

In [ ]:
import numpy as np

def blend(preds, weights=None):
    """preds: list array sama bentuk. Bekerja untuk probabilitas, nilai regresi, dan bagging seed."""
    return np.average(np.stack(preds), axis=0, weights=weights)

import numpy as np
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

def best_threshold(y, p, grid=None, smooth=5):
    """p: probabilitas kelas positif. Kurva dihaluskan agar tidak mengejar noise."""
    grid = np.linspace(.05, .95, 91) if grid is None else grid
    f = np.array([f1_score(y, p >= t, zero_division=0) for t in grid])
    return float(grid[np.convolve(f, np.ones(smooth) / smooth, mode="same").argmax()])

def nested_threshold_score(y, p, n=5, seed=42):
    """Skor jujur: threshold difit di fold lain. Kembalikan (F1 nested, threshold seluruh OOF)."""
    y, p = np.asarray(y), np.asarray(p)
    pred = np.zeros(len(y), int)
    for a, b in StratifiedKFold(n, shuffle=True, random_state=seed).split(p, y):
        pred[b] = p[b] >= best_threshold(y[a], p[a])
    return round(f1_score(y, pred, zero_division=0), 4), best_threshold(y, p)

import numpy as np
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

def fit_multipliers(y, p, grid=np.linspace(.5, 2.0, 16), rounds=2):
    """Coordinate ascent. Prediksi = argmax(p * w)."""
    y, p = np.asarray(y), np.asarray(p)
    w = np.ones(p.shape[1])
    best = f1_score(y, (p * w).argmax(1), average="macro", zero_division=0)
    for _ in range(rounds):
        for k in range(p.shape[1]):
            for g in grid:
                c = w.copy(); c[k] = g
                s = f1_score(y, (p * c).argmax(1), average="macro", zero_division=0)
                if s > best + 1e-5:
                    best, w = s, c
    return w

def nested_multiplier_score(y, p, n=5, seed=42):
    """Kembalikan (macro-F1 argmax, macro-F1 nested, pengali seluruh OOF). Pakai bila nested lebih tinggi melebihi noise."""
    y, p = np.asarray(y), np.asarray(p)
    pred = np.zeros(len(y), int)
    for a, b in StratifiedKFold(n, shuffle=True, random_state=seed).split(p, y):
        pred[b] = (p[b] * fit_multipliers(y[a], p[a])).argmax(1)
    return round(f1_score(y, p.argmax(1), average="macro"), 4), round(f1_score(y, pred, average="macro"), 4), fit_multipliers(y, p)

def fit_label_thresholds(Y, P, grid=np.linspace(.05, .95, 19)):
    Y, P = np.asarray(Y), np.asarray(P)
    return np.array([grid[np.argmax([f1_score(Y[:, j], P[:, j] >= t, zero_division=0) for t in grid])] for j in range(Y.shape[1])])

def apply_thresholds(P, t, at_least_one=False):
    P = np.asarray(P)
    out = (P >= t).astype(int)
    if at_least_one:
        none = out.sum(1) == 0
        out[none, P[none].argmax(1)] = 1
    return out

# ===== jalankan =====
if bert_valid is not None:
    va, vp = bert_valid
    g = paired_gain(y[va], oof[va].argmax(1), blend([oof[va], vp]).argmax(1), macro)
    print("gain blend", g)
    if g["gain"] > 2 * g["se"]:
        pred = blend([pred, bert_test])
if len(classes) == 2:
    f_nested, thr = nested_threshold_score(y[COV], oof[COV, 1])
    print("F1 nested", f_nested, "| threshold", thr)
    final = (pred[:, 1] >= thr).astype(int)
else:
    pp = COV[sample_idx(len(COV), 100000, SEED)]
    base, nested, w = nested_multiplier_score(y[pp], oof[pp])
    print("macro-F1 argmax", base, "| nested", nested)
    final = (pred * w).argmax(1) if nested > base + .002 else pred.argmax(1)


## Submission & grafik

Validasi kolom, jumlah baris, NaN, dan nilai label. Gambar untuk slide.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix

def _save(path):
    plt.tight_layout(); plt.savefig(path, dpi=160); plt.close()
    return path

def plot_heatmap(table, path="heatmap.png", title="Aspek x sentimen", fmt=".0f", normalize=False):
    t = table.div(table.sum(1), axis=0) if normalize else table
    plt.figure(figsize=(8, max(3, .4 * len(t))))
    sns.heatmap(t, annot=True, fmt=".2f" if normalize else fmt, cmap="YlGnBu", cbar=False)
    plt.title(title)
    return _save(path)

def plot_confusion(y, pred, names=None, path="confusion.png"):
    c = confusion_matrix(y, pred, normalize="true")
    return plot_heatmap(pd.DataFrame(c, index=names, columns=names), path, "Confusion matrix (per baris)", fmt=".2f")

def plot_timeline(ts, path="timeline.png", events=None, freq="W", title="Volume per minggu"):
    """events: {'YYYY-MM-DD': 'nama kejadian'}"""
    t = pd.to_datetime(pd.Series(list(ts)), errors="coerce").dropna()
    v = t.dt.to_period(freq).value_counts().sort_index()
    x = v.index.to_timestamp()
    plt.figure(figsize=(10, 3.6)); plt.plot(x, v.values, color="#0f766e")
    for d, name in (events or {}).items():
        plt.axvline(pd.Timestamp(d), color="#b42318", ls="--", lw=1); plt.text(pd.Timestamp(d), v.max(), name, fontsize=8, rotation=90, va="top")
    plt.title(title)
    return _save(path)

def plot_bars(series, path="bars.png", title="Distribusi", top=20):
    s = pd.Series(series).sort_values().tail(top)
    plt.figure(figsize=(8, max(3, .35 * len(s)))); plt.barh(s.index.astype(str), s.values, color="#0f766e"); plt.title(title)
    return _save(path)

# ===== jalankan =====
sub = write_submission(SAMPLE, classes[final], allowed=classes)
print(plot_confusion(y[COV], oof[COV].argmax(1), list(classes)))
print(leaderboard().to_string())
clock("submission")


## Bahan slide MBG

Heatmap aspek x sentimen, lokasi, dan tren dari label train dan prediksi test. Baca audit sebelum dipakai: kamus adalah hipotesis.

In [ ]:
import re
import ahocorasick
import numpy as np
import pandas as pd

MBG_ENTITIES = {
    "MBG": ["mbg", "makan bergizi gratis", "makan bergizi", "makan gratis", "makan siang gratis", "program makan gratis", "embege"],
    "BGN": ["bgn", "badan gizi nasional", "badan gizi"],
    "SPPG": ["sppg", "satuan pelayanan pemenuhan gizi", "dapur mbg", "dapur sppg"],
    "Kemenkes": ["kemenkes", "kementerian kesehatan", "menkes"], "Kejagung": ["kejagung", "kejaksaan agung"], "KPK": ["kpk"],
    "DPR": ["dpr", "banggar"], "JPPI": ["jppi"], "KPAI": ["kpai"], "Ombudsman": ["ombudsman"], "ICW": ["icw"], "CISDI": ["cisdi"],
    "Prabowo": ["prabowo", "wowo"], "Gibran": ["gibran"], "Sudaryono": ["sudaryono"], "Dadan Hindayana": ["dadan", "dadan hindayana"], "Nanik Deyang": ["nanik"], "Purbaya": ["purbaya"],
}
MBG_ASPECTS = {
    "keamanan_pangan": ["keracunan", "keracunannya", "racun", "beracun", "basi", "busuk", "berjamur", "jamuran", "belatung", "ulat", "ulet", "klb", "e coli", "salmonella", "kadaluarsa", "kedaluwarsa", "tidak layak", "gak layak", "ga layak", "mual", "muntah", "diare", "pusing", "sakit perut", "dilarikan", "puskesmas", "rumah sakit", "korban", "bau", "mentah"],
    "kualitas_menu": ["menu", "ompreng", "ayam", "nasi", "sayur", "susu", "burger", "spaghetti", "spageti", "roti", "telur", "telor", "tempe", "ikan", "kacang", "buah", "pisang", "jeruk", "lauk", "enak", "hambar", "porsi", "ultra processed", "upf", "sosis", "nugget"],
    "gizi_kesehatan": ["gizi", "bergizi", "ahli gizi", "nutrisi", "stunting", "protein", "kalori", "akg", "obesitas", "sehat", "kesehatan", "dokter"],
    "anggaran": ["anggaran", "apbn", "rapbn", "pajak", "triliun", "pagu", "efisiensi", "pemborosan", "refocusing", "dana", "buang uang", "uang rakyat", "biaya", "per porsi", "15 ribu", "10 ribu", "10rb", "15rb"],
    "korupsi_tata_kelola": ["korupsi", "koruptor", "korup", "nepotisme", "kolusi", "markup", "mark up", "tersangka", "kejagung", "jual beli titik", "yayasan", "calo", "transparan", "transparansi", "proyek", "bancakan", "berjamaah", "kpk", "tata kelola", "pungli"],
    "distribusi_dapur": ["sppg", "dapur", "distribusi", "pengiriman", "terlambat", "telat", "kapasitas", "suspend", "sanitasi", "slhs", "sop", "relawan", "mitra", "juknis", "ditutup", "penerima", "belum dapat", "tidak merata", "gak merata", "merata"],
    "pendidikan_guru": ["guru", "honorer", "murid", "siswa", "sekolah", "kantin", "santri", "pendidikan", "gaji guru", "biaya sekolah", "sekolah gratis"],
    "ekonomi_umkm": ["umkm", "lapangan kerja", "pekerja", "pedagang", "jualan", "catering", "katering", "petani", "peternak", "nelayan", "susu lokal"],
    "politik": ["prabowo", "wowo", "gibran", "dpr", "tni", "polri", "janji kampanye", "militer", "purbaya", "zulhas", "sudaryono", "nanik", "dadan", "presiden", "pemerintah", "menteri"],
    "media_hoaks": ["hoax", "hoaks", "buzzer", "influencer", "narasi", "digoreng", "komdigi", "klarifikasi", "fitnah"],
    "penolakan": ["stop", "setop", "hentikan", "bubar", "bubarkan", "batalkan", "hapus", "ganti uang", "tolak", "menolak"],
    "minta_perbaikan": ["evaluasi", "perbaiki", "perbaikan", "benahi", "awasi", "diawasi", "pengawasan", "tolong", "seharusnya", "harusnya"],
    "dukungan": ["bagus", "setuju", "mendukung", "dukung", "program bagus", "bermanfaat", "terima kasih", "makasih", "alhamdulillah", "lanjutkan", "semoga"],
}
MBG_EVENTS = {"2025-01-06": "Peluncuran MBG", "2025-09-24": "KLB Bandung Barat", "2026-06-02": "Pimpinan BGN dicopot", "2026-07-22": "Kepala BGN baru", "2026-10-29": "Laporan 100 hari"}
MBG_FOOD = sorted({w for w in MBG_ASPECTS["kualitas_menu"] if w not in ("menu", "ompreng", "enak", "hambar", "porsi", "ultra processed", "upf")})
MBG_SYMPTOM = ["mual", "muntah", "diare", "pusing", "sakit perut", "demam", "lemas"]
MBG_INCIDENT = ["keracunan", "basi", "busuk", "berjamur", "belatung", "ulat", "klb", "kadaluarsa", "kedaluwarsa"]

_SUFFIX = re.compile(r"(?:nya|ny|ku|mu|lah|kah)?(?![0-9a-z])")

def build_matcher(lex):
    """lex: {kelas: [istilah]}. Kembalikan automaton, dipakai find_terms."""
    a = ahocorasick.Automaton()
    for key, terms in lex.items():
        for t in terms:
            t = t.lower()
            old = a.get(t) if t in a else (t, [])
            a.add_word(t, (t, old[1] + [key]))
    a.make_automaton()
    return a

def find_terms(text, matcher):
    """Istilah utuh dalam teks. Akhiran -nya, -ku, -mu ikut cocok: 'menunya' cocok 'menu'."""
    t = str(text).lower()
    out = []
    for end, (term, keys) in matcher.iter(t):
        start = end - len(term) + 1
        if (start == 0 or not t[start - 1].isalnum()) and _SUFFIX.match(t, end + 1):
            out.append((term, keys))
    return out

def tag_terms(texts, lex=None):
    """DataFrame bool: baris x kelas. Teks identik hanya diproses sekali."""
    lex = lex or MBG_ASPECTS
    keys, idx = list(lex), {k: i for i, k in enumerate(lex)}
    m = build_matcher(lex)
    texts = ["" if t is None or t != t else str(t) for t in texts]
    cache = {}
    for t in set(texts):
        row = np.zeros(len(keys), dtype=bool)
        for _, ks in find_terms(t, m):
            row[[idx[k] for k in ks]] = True
        cache[t] = row
    return pd.DataFrame(np.vstack([cache[t] for t in texts]) if texts else np.zeros((0, len(keys)), bool), columns=keys)

def term_hits(texts, lex=None):
    """Jumlah baris per istilah. Istilah nol hit dibuang, istilah terlalu sering diaudit."""
    lex = lex or MBG_ASPECTS
    m = build_matcher(lex)
    c = {}
    for t in texts:
        for term in {x[0] for x in find_terms(t, m)}:
            c[term] = c.get(term, 0) + 1
    rows = [(k, t, c.get(t.lower(), 0)) for k, v in lex.items() for t in v]
    return pd.DataFrame(rows, columns=["kelas", "istilah", "hit"]).sort_values(["kelas", "hit"], ascending=[True, False]).reset_index(drop=True)

def audit_aspects(texts, H, n=30, seed=42):
    """{kelas: contoh acak}. Baca dan hitung presisinya sebelum dipakai di slide."""
    texts = pd.Series(list(texts))
    return {k: texts[H[k].to_numpy()].drop_duplicates().sample(min(n, int(H[k].sum())), random_state=seed).tolist() for k in H if H[k].any()}

def aspect_sentiment(H, sentiment):
    """Tabel aspek x sentimen. Satu baris dihitung di semua aspeknya."""
    s = np.asarray(sentiment)
    return pd.DataFrame({k: pd.Series(s[H[k].to_numpy()]).value_counts() for k in H}).T.fillna(0).astype(int)

import re
from pathlib import Path
import ahocorasick
import numpy as np
import pandas as pd

WILAYAH = {
    "Aceh": "Aceh Barat|Aceh Barat Daya|Aceh Besar|Aceh Jaya|Aceh Selatan|Aceh Singkil|Aceh Tamiang|Aceh Tengah|Aceh Tenggara|Aceh Timur|Aceh Utara|Banda Aceh|Bener Meriah|Bireuen|Gayo Lues|Langsa|Lhokseumawe|Nagan Raya|Pidie|Pidie Jaya|Sabang|Simeulue|Subulussalam",
    "Bali": "Badung|Bangli|Buleleng|Denpasar|Gianyar|Jembrana|Karangasem|Klungkung|Tabanan",
    "Banten": "Cilegon|Lebak|Pandeglang|Serang|Tangerang|Tangerang Selatan",
    "Bengkulu": "Bengkulu|Bengkulu Selatan|Bengkulu Tengah|Bengkulu Utara|Kaur|Kepahiang|Lebong|Mukomuko|Rejang Lebong|Seluma",
    "Daerah Istimewa Yogyakarta": "Bantul|Gunungkidul|Kulon Progo|Sleman|Yogyakarta",
    "Daerah Khusus Ibukota Jakarta": "Jakarta Barat|Jakarta Pusat|Jakarta Selatan|Jakarta Timur|Jakarta Utara|Kepulauan Seribu",
    "Gorontalo": "Boalemo|Bone Bolango|Gorontalo|Gorontalo Utara|Pohuwato",
    "Jambi": "Batanghari|Bungo|Jambi|Kerinci|Merangin|Muaro Jambi|Sarolangun|Sungai Penuh|Tanjung Jabung Barat|Tanjung Jabung Timur|Tebo",
    "Jawa Barat": "Bandung|Bandung Barat|Banjar|Bekasi|Bogor|Ciamis|Cianjur|Cimahi|Cirebon|Depok|Garut|Indramayu|Karawang|Kuningan|Majalengka|Pangandaran|Purwakarta|Subang|Sukabumi|Sumedang|Tasikmalaya",
    "Jawa Tengah": "Banjarnegara|Banyumas|Batang|Blora|Boyolali|Brebes|Cilacap|Demak|Grobogan|Jepara|Karanganyar|Kebumen|Kendal|Klaten|Kudus|Magelang|Pati|Pekalongan|Pemalang|Purbalingga|Purworejo|Rembang|Salatiga|Semarang|Sragen|Sukoharjo|Surakarta|Tegal|Temanggung|Wonogiri|Wonosobo",
    "Jawa Timur": "Bangkalan|Banyuwangi|Batu|Blitar|Bojonegoro|Bondowoso|Gresik|Jember|Jombang|Kediri|Lamongan|Lumajang|Madiun|Magetan|Malang|Mojokerto|Nganjuk|Ngawi|Pacitan|Pamekasan|Pasuruan|Ponorogo|Probolinggo|Sampang|Sidoarjo|Situbondo|Sumenep|Surabaya|Trenggalek|Tuban|Tulungagung",
    "Kalimantan Barat": "Bengkayang|Kapuas Hulu|Kayong Utara|Ketapang|Kubu Raya|Landak|Melawi|Mempawah|Pontianak|Sambas|Sanggau|Sekadau|Singkawang|Sintang",
    "Kalimantan Selatan": "Balangan|Banjar|Banjarbaru|Banjarmasin|Barito Kuala|Hulu Sungai Selatan|Hulu Sungai Tengah|Hulu Sungai Utara|Kotabaru|Tabalong|Tanah Bumbu|Tanah Laut|Tapin",
    "Kalimantan Tengah": "Barito Selatan|Barito Timur|Barito Utara|Gunung Mas|Kapuas|Katingan|Kotawaringin Barat|Kotawaringin Timur|Lamandau|Murung Raya|Palangkaraya|Pulang Pisau|Seruyan|Sukamara",
    "Kalimantan Timur": "Balikpapan|Berau|Bontang|Kutai Barat|Kutai Kartanegara|Kutai Timur|Mahakam Ulu|Paser|Penajam Paser Utara|Samarinda",
    "Kalimantan Utara": "Bulungan|Malinau|Nunukan|Tana Tidung|Tarakan",
    "Kepulauan Bangka Belitung": "Bangka|Bangka Barat|Bangka Selatan|Bangka Tengah|Belitung|Belitung Timur|Pangkal Pinang",
    "Kepulauan Riau": "Batam|Bintan|Karimun|Kepulauan Anambas|Lingga|Natuna|Tanjung Pinang",
    "Lampung": "Bandar Lampung|Lampung Barat|Lampung Selatan|Lampung Tengah|Lampung Timur|Lampung Utara|Mesuji|Metro|Pesawaran|Pesisir Barat|Pringsewu|Tanggamus|Tulang Bawang|Tulang Bawang Barat|Way Kanan",
    "Maluku": "Ambon|Buru|Buru Selatan|Kepulauan Aru|Kepulauan Tanimbar|Maluku Barat Daya|Maluku Tengah|Maluku Tenggara|Seram Bagian Barat|Seram Bagian Timur|Tual",
    "Maluku Utara": "Halmahera Barat|Halmahera Selatan|Halmahera Tengah|Halmahera Timur|Halmahera Utara|Kepulauan Sula|Pulau Morotai|Pulau Taliabu|Ternate|Tidore Kepulauan",
    "Nusa Tenggara Barat": "Bima|Dompu|Lombok Barat|Lombok Tengah|Lombok Timur|Lombok Utara|Mataram|Sumbawa|Sumbawa Barat",
    "Nusa Tenggara Timur": "Alor|Belu|Ende|Flores Timur|Kab Timor Tengah Selatan|Kupang|Lembata|Malaka|Manggarai|Manggarai Barat|Manggarai Timur|Nagekeo|Ngada|Rote Ndao|Sabu Raijua|Sikka|Sumba Barat|Sumba Barat Daya|Sumba Tengah|Sumba Timur|Timor Tengah Utara",
    "Papua": "Biak Numfor|Jayapura|Keerom|Kepulauan Yapen|Mamberamo Raya|Sarmi|Supiori|Waropen",
    "Papua Barat": "Fak Fak|Kaimana|Manokwari|Manokwari Selatan|Pegunungan Arfak|Teluk Bintuni|Teluk Wondama",
    "Papua Barat Daya": "Maybrat|Raja Ampat|Sorong|Sorong Selatan|Tambrauw",
    "Papua Pegunungan": "Jayawijaya|Kab Pegunungan Bintang|Lanny Jaya|Mamberamo Tengah|Nduga|Tolikara|Yahukimo|Yalimo",
    "Papua Selatan": "Asmat|Boven Digoel|Mappi|Merauke",
    "Papua Tengah": "Deiyai|Dogiyai|Intan Jaya|Mimika|Nabire|Paniai|Puncak|Puncak Jaya",
    "Riau": "Bengkalis|Dumai|Indragiri Hilir|Indragiri Hulu|Kampar|Kepulauan Meranti|Kuantan Singingi|Pekanbaru|Pelalawan|Rokan Hilir|Rokan Hulu|Siak",
    "Sulawesi Barat": "Majene|Mamasa|Mamuju|Mamuju Tengah|Pasangkayu|Polewali Mandar",
    "Sulawesi Selatan": "Bantaeng|Barru|Bone|Bulukumba|Enrekang|Gowa|Jeneponto|Kepulauan Selayar|Luwu|Luwu Timur|Luwu Utara|Makassar|Maros|Palopo|Pangkajene dan Kepulauan|Parepare|Pinrang|Sidenreng Rappang|Sinjai|Soppeng|Takalar|Tana Toraja|Toraja Utara|Wajo",
    "Sulawesi Tengah": "Banggai|Banggai Kepulauan|Banggai Laut|Buol|Donggala|Morowali|Morowali Utara|Palu|Parigi Moutong|Poso|Sigi|Tojo Una Una|Toli-Toli",
    "Sulawesi Tenggara": "Bau Bau|Bombana|Buton|Buton Selatan|Buton Tengah|Buton Utara|Kendari|Kolaka|Kolaka Timur|Kolaka Utara|Konawe|Konawe Kepulauan|Konawe Selatan|Konawe Utara|Muna|Muna Barat|Wakatobi",
    "Sulawesi Utara": "Bitung|Bolaang Mongondow|Bolaang Mongondow Selatan|Bolaang Mongondow Timur|Bolaang Mongondow Utara|Kep. Siau Tagulandang Biaro|Kepulauan Sangihe|Kepulauan Talaud|Kotamobagu|Manado|Minahasa|Minahasa Selatan|Minahasa Tenggara|Minahasa Utara|Tomohon",
    "Sumatera Barat": "Agam|Bukittinggi|Dharmasraya|Kepulauan Mentawai|Lima Puluh Kota|Padang|Padang Panjang|Padang Pariaman|Pariaman|Pasaman|Pasaman Barat|Payakumbuh|Pesisir Selatan|Sawahlunto|Sijunjung|Solok|Solok Selatan|Tanah Datar",
    "Sumatera Selatan": "Banyuasin|Empat Lawang|Lahat|Lubuk Linggau|Muara Enim|Musi Banyuasin|Musi Rawas|Musi Rawas Utara|Ogan Ilir|Ogan Komering Ilir|Ogan Komering Ulu|Ogan Komering Ulu Selatan|Ogan Komering Ulu Timur|Pagar Alam|Palembang|Penukal Abab Lematang Ilir|Prabumulih",
    "Sumatera Utara": "Asahan|Batu Bara|Binjai|Dairi|Deli Serdang|Gunungsitoli|Humbang Hasundutan|Karo|Labuhanbatu|Labuhanbatu Selatan|Labuhanbatu Utara|Langkat|Mandailing Natal|Medan|Nias|Nias Barat|Nias Selatan|Nias Utara|Padang Lawas|Padang Lawas Utara|Padangsidimpuan|Pakpak Bharat|Pematangsiantar|Samosir|Serdang Bedagai|Sibolga|Simalungun|Tanjungbalai|Tapanuli Selatan|Tapanuli Tengah|Tapanuli Utara|Tebing Tinggi|Toba"
}
# Provinsi tidak punya baris di atas sebagai nama; alias umum di media sosial.
PROV_ALIAS = {"Aceh": "aceh", "Bali": "bali", "Banten": "banten", "Daerah Istimewa Yogyakarta": "diy|jogja|jogjakarta|yogya|yogyakarta", "Daerah Khusus Ibukota Jakarta": "dki|jakarta|jkt",
              "Jawa Barat": "jawa barat|jabar", "Jawa Tengah": "jawa tengah|jateng", "Jawa Timur": "jawa timur|jatim", "Kalimantan Barat": "kalimantan barat|kalbar", "Kalimantan Selatan": "kalimantan selatan|kalsel",
              "Kalimantan Tengah": "kalimantan tengah|kalteng", "Kalimantan Timur": "kalimantan timur|kaltim", "Kalimantan Utara": "kalimantan utara|kaltara", "Kepulauan Bangka Belitung": "bangka belitung|babel",
              "Kepulauan Riau": "kepulauan riau|kepri", "Lampung": "lampung", "Maluku": "maluku", "Maluku Utara": "maluku utara|malut", "Nusa Tenggara Barat": "nusa tenggara barat|ntb", "Nusa Tenggara Timur": "nusa tenggara timur|ntt",
              "Papua": "papua", "Papua Barat": "papua barat", "Papua Barat Daya": "papua barat daya", "Papua Pegunungan": "papua pegunungan", "Papua Selatan": "papua selatan", "Papua Tengah": "papua tengah", "Riau": "riau",
              "Sulawesi Barat": "sulawesi barat|sulbar", "Sulawesi Selatan": "sulawesi selatan|sulsel", "Sulawesi Tengah": "sulawesi tengah|sulteng", "Sulawesi Tenggara": "sulawesi tenggara|sultra", "Sulawesi Utara": "sulawesi utara|sulut",
              "Sumatera Barat": "sumatera barat|sumbar", "Sumatera Selatan": "sumatera selatan|sumsel", "Sumatera Utara": "sumatera utara|sumut", "Gorontalo": "gorontalo", "Bengkulu": "bengkulu", "Jambi": "jambi"}
KAB_ALIAS = {"bdg": "Bandung", "kbb": "Bandung Barat", "smg": "Semarang", "solo": "Surakarta", "mks": "Makassar", "plg": "Palembang", "tangsel": "Tangerang Selatan", "jaksel": "Jakarta Selatan", "jakbar": "Jakarta Barat", "jaktim": "Jakarta Timur", "jakut": "Jakarta Utara", "jakpus": "Jakarta Pusat"}
# Nama yang juga kata umum: hanya dihitung bila didahului konteks lokasi.
AMBIGUOUS = set("""padang batu malang serang medan buru sabang merauke bone metro bima ketapang karo kudus puncak batang bangka banjar lebak tegal kediri sampang luwu asmat agam pati kaur alor muna solok landak
belu ende rote siak bungo sigi poso dairi balangan tojo gowa deli maros wajo pinrang soppeng barru bantaeng takalar selayar sinjai sidrap tana lingga bintan natuna anambas tolitoli buol parigi morowali konawe kolaka
solo ngawi nganjuk sumba flores sikka ngada lembata kupang timor seram tual aru dobo sula tidore ternate halmahera morotai biak nabire mimika paniai dogiyai deiyai yalimo tolikara lanny nduga puncak yahukimo boven mappi
ambon sarmi keerom waropen supiori kaimana fakfak bintuni wondama maybrat tambrauw sorong raja ampat""".split())
_CONTEXT = re.compile(r"(?:\b(?:di|ke|dari|kab|kabupaten|kota|kec|kecamatan|daerah|wilayah|warga|sppg|sd|smp|sma|smk|sdn|smpn|sman|smkn|mts|man|mi|pemkab|pemkot|bupati|walikota|wali kota)\s+)$")
_KAB = {n.lower(): (n, p) for p, s in WILAYAH.items() for n in s.split("|")}
_KAB.update({a: _KAB[n.lower()] for a, n in KAB_ALIAS.items() if n.lower() in _KAB})
_PROV = {a: p for p, s in PROV_ALIAS.items() for a in s.split("|")}
_AUTO = ahocorasick.Automaton()
for _k in set(_KAB) | set(_PROV):
    _AUTO.add_word(_k, _k)
_AUTO.make_automaton()

def find_locations(text):
    """[(kabupaten/kota atau None, provinsi)]. Pencocokan terpanjang: 'bandung barat' mengalahkan 'bandung'."""
    t = str(text).lower()
    hits = []
    for end, k in _AUTO.iter(t):
        s = end - len(k) + 1
        if (s == 0 or not t[s - 1].isalnum()) and (end + 1 == len(t) or not t[end + 1].isalnum()):
            hits.append((s, end, k))
    hits.sort(key=lambda h: (h[0], h[0] - h[1]))
    out, last = [], -1
    for s, e, k in sorted(hits, key=lambda h: h[0] - h[1]):
        if any(not (e < a or s > b) for a, b in [(x[0], x[1]) for x in out]):
            continue
        if k in AMBIGUOUS and not _CONTEXT.search(t[max(0, s - 22):s]):
            continue
        out.append((s, e, k))
    res = []
    for s, e, k in sorted(out):
        if k in _KAB:
            res.append(_KAB[k])
        else:
            res.append((None, _PROV[k]))
    return res

def tag_locations(texts):
    """DataFrame: kab_kota, provinsi (lokasi pertama yang paling spesifik), n_lokasi. Teks identik sekali."""
    texts = ["" if t is None or t != t else str(t) for t in texts]
    cache = {}
    for t in set(texts):
        r = find_locations(t)
        kab = [x for x in r if x[0]]
        first = kab[0] if kab else r[0] if r else (None, None)
        cache[t] = (first[0], first[1], len(r))
    return pd.DataFrame([cache[t] for t in texts], columns=["kab_kota", "provinsi", "n_lokasi"])

def resolve_location(name, cutoff=85):
    """Nama dari NER atau kolom metadata ke nama baku, tahan typo. Kembalikan (kab_kota, provinsi, skor)."""
    from rapidfuzz import fuzz, process
    n = re.sub(r"^(kabupaten|kab\.?|kota|provinsi|prov\.?)\s+", "", str(name).lower().strip())
    if n in _KAB:
        return (*_KAB[n], 100.0)
    if n in _PROV:
        return (None, _PROV[n], 100.0)
    m = process.extractOne(n, list(_KAB) + list(_PROV), scorer=fuzz.WRatio, score_cutoff=cutoff)
    if m is None:
        return (None, None, 0.0)
    return (*_KAB[m[0]], float(m[1])) if m[0] in _KAB else (None, _PROV[m[0]], float(m[1]))

# Nama pada geoBoundaries ADM1 (2017, 34 provinsi): empat provinsi Papua baru digabung ke induknya.
_ADM1 = {"Bali": "Bali", "Nusa Tenggara Barat": "West Nusa Tenggara", "Banten": "Banten", "Jawa Tengah": "Central Java", "Jawa Barat": "West Java", "Kalimantan Tengah": "Central Kalimantan", "Kalimantan Selatan": "South Kalimantan", "Kalimantan Barat": "West Kalimantan",
         "Sulawesi Tengah": "Central Sulawesi", "Gorontalo": "Gorontalo", "Sulawesi Utara": "North Sulawesi", "Sulawesi Selatan": "South Sulawesi", "Sulawesi Tenggara": "Southeast Sulawesi", "Sulawesi Barat": "West Sulawesi", "Aceh": "Aceh", "Bengkulu": "Bengkulu", "Jambi": "Jambi",
         "Lampung": "Lampung", "Riau": "Riau", "Sumatera Barat": "West Sumatra", "Sumatera Selatan": "South Sumatra", "Sumatera Utara": "North Sumatra", "Nusa Tenggara Timur": "East Nusa Tenggara", "Maluku": "Maluku", "Maluku Utara": "North Maluku", "Jawa Timur": "East Java",
         "Kepulauan Bangka Belitung": "Bangka-Belitung Islands", "Kepulauan Riau": "Riau Islands", "Papua": "Papua", "Papua Selatan": "Papua", "Papua Tengah": "Papua", "Papua Pegunungan": "Papua", "Papua Barat": "West Papua", "Papua Barat Daya": "West Papua",
         "Kalimantan Timur": "East Kalimantan", "Kalimantan Utara": "North Kalimantan", "Daerah Istimewa Yogyakarta": "Special Region of Yogyakarta", "Daerah Khusus Ibukota Jakarta": "Jakarta Special Capital Region"}

def plot_province_map(counts, geojson=None, path="peta.png", title="Sebaran per provinsi"):
    """counts: Series provinsi -> nilai. Tanpa file geojson atau geopandas, otomatis jadi grafik batang."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    s = pd.Series(counts).astype(float)
    try:
        if not geojson or not Path(geojson).exists():
            raise FileNotFoundError(geojson)
        import geopandas as gpd
        g = gpd.read_file(geojson)
        v = s.groupby(s.index.map(lambda p: _ADM1.get(p, p))).sum()
        g["nilai"] = g.shapeName.map(v).fillna(0)
        ax = g.plot(column="nilai", cmap="YlOrRd", legend=True, figsize=(11, 4.6), edgecolor="#667085", linewidth=.3)
        ax.set_axis_off()
    except Exception as e:
        print("peta dilewati, pakai grafik batang:", type(e).__name__)
        t = s.sort_values().tail(20)
        plt.figure(figsize=(8, max(3, .35 * len(t))))
        plt.barh(t.index.astype(str), t.values, color="#b42318")
    plt.title(title)
    plt.tight_layout(); plt.savefig(path, dpi=160); plt.close()
    return path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ruptures as rpt

def _period(times, freq):
    t = pd.to_datetime(pd.Series(list(times)), errors="coerce", utc=True, format="mixed").dt.tz_localize(None)
    return t, t.dt.to_period(freq)

def issue_timeline(times, mask=None, freq="W", min_total=30):
    """Tabel per periode: total, isu, proporsi. Periode di bawah min_total ditandai agar tidak dibaca."""
    t, p = _period(times, freq)
    m = np.ones(len(t), bool) if mask is None else np.asarray(mask, bool)
    tl = pd.DataFrame({"total": p[t.notna()].value_counts(), "isu": p[t.notna() & m].value_counts()}).fillna(0).astype(int).sort_index()
    tl["proporsi"] = (tl.isu / tl.total).round(4)
    tl["cukup"] = tl.total >= min_total
    return tl

def change_points(tl, col="proporsi", penalty=None, min_size=2):
    """Titik perubahan pada periode yang cukup datanya. Kembalikan daftar periode awal segmen baru."""
    s = tl[tl.cukup][col] if "cukup" in tl else tl[col]
    if len(s) < 2 * min_size + 1:
        return []
    x = s.to_numpy(float).reshape(-1, 1)
    pen = penalty if penalty is not None else 3 * np.log(len(x)) * max(float(np.var(x)), 1e-9)
    bk = rpt.Pelt(model="l2", min_size=min_size, jump=1).fit(x).predict(pen=pen)
    return [s.index[i] for i in bk[:-1]]

def share_over_time(times, labels, freq="M", min_total=30):
    """Proporsi tiap topik per periode. Periode kecil dibuang."""
    t, p = _period(times, freq)
    ok = t.notna().to_numpy()
    c = pd.crosstab(p[ok].to_numpy(), np.asarray(labels)[ok])
    c = c[c.sum(1) >= min_total]
    return c.div(c.sum(1), axis=0).round(3)

def plot_issue_timeline(tl, path="tren_isu.png", title="Porsi isu per periode", events=None, cps=None):
    d = tl[tl.cukup] if "cukup" in tl else tl
    x = d.index.to_timestamp()
    fig, ax = plt.subplots(figsize=(10, 3.8))
    ax.bar(x, d.total, width=(x[1] - x[0]) * .8 if len(x) > 1 else 1, color="#d0d5dd")
    ax.set_ylabel("volume")
    a2 = ax.twinx()
    a2.plot(x, d.proporsi, color="#b42318", lw=2)
    a2.set_ylabel("porsi isu")
    for c in cps or []:
        ax.axvline(c.to_timestamp(), color="#0f766e", lw=1.5)
    for when, name in (events or {}).items():
        w = pd.Timestamp(when)
        if len(x) and x[0] <= w <= x[-1]:
            ax.axvline(w, color="#667085", ls="--", lw=1)
            ax.text(w, d.total.max(), name, fontsize=8, rotation=90, va="top")
    ax.set_title(title)
    plt.tight_layout(); plt.savefig(path, dpi=160); plt.close()
    return path

def bertopic_over_time(docs, times, embeddings, nr_bins=12, min_topic_size=30):
    """BERTopic pada embedding yang sudah ada. Pakai sampel 20-50 ribu baris; RAM 500 ribu belum teruji."""
    from bertopic import BERTopic
    tm = BERTopic(min_topic_size=min_topic_size, calculate_probabilities=False, low_memory=True, verbose=False)
    tm.fit(docs, embeddings)
    return tm, tm.topics_over_time(docs, list(pd.to_datetime(pd.Series(list(times)), errors="coerce", utc=True, format="mixed").dt.tz_localize(None)), nr_bins=nr_bins)

from pptx import Presentation
from pptx.util import Inches, Pt

def build_slides(slides, out="hasil.pptx", template=None):
    """slides: list dict {title, bullets, image}. Satu slide per dict."""
    prs = Presentation(template) if template else Presentation()
    for s in slides:
        sl = prs.slides.add_slide(prs.slide_layouts[1])
        sl.shapes.title.text = s.get("title", "")
        body = sl.placeholders[1].text_frame
        body.clear()
        for i, b in enumerate(s.get("bullets", [])):
            p = body.paragraphs[0] if i == 0 else body.add_paragraph()
            p.text = str(b); p.font.size = Pt(20)
        if s.get("image"):
            sl.shapes.add_picture(s["image"], Inches(.6), Inches(3.2), height=Inches(3.8))
    prs.save(out)
    return out

# ===== jalankan =====
if MBG_INSIGHT:
    all_text = pd.concat([train[TEXT], test[TEXT]], ignore_index=True)
    all_sent = np.concatenate([np.asarray(classes)[y], np.asarray(classes)[final]])   # label train + prediksi test
    H = tag_terms(all_text)
    print(H.mean().round(3).to_string())
    print("istilah nol hit:", term_hits(all_text.sample(min(50000, len(all_text)), random_state=SEED)).query("hit == 0").istilah.tolist()[:40])
    tbl = aspect_sentiment(H, all_sent)
    print(tbl.to_string())
    imgs = {"Aspek x sentimen": plot_heatmap(tbl, "aspek_sentimen.png", normalize=True)} if tbl.to_numpy().sum() else {}
    L = tag_locations(all_text)
    if L.provinsi.notna().any():
        imgs["Sebutan provinsi"] = plot_province_map(L.provinsi.value_counts(), MODELS.parent / "data_ext/wilayah/idn_adm1.geojson", "peta.png", "Provinsi yang disebut")
    if TIME and TIME in train and TIME in test:
        times = pd.concat([train[TIME], test[TIME]], ignore_index=True)
        tl = issue_timeline(times, H["keamanan_pangan"], "W")
        imgs["Tren keamanan pangan"] = plot_issue_timeline(tl, "tren.png", "Porsi keamanan pangan per minggu", MBG_EVENTS, change_points(tl))
    if imgs:
        build_slides([{"title": k, "bullets": [], "image": v} for k, v in imgs.items()], "bahan_slide.pptx")
    print("gambar:", list(imgs.values()))
else:
    print("MBG_INSIGHT=False")
